In [ ]:
# Cell 1 - run information (required)
import subprocess, sys, random, numpy as np, torch
SEED = 266; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True,
                        text=True).stdout.strip()
print({"commit": commit, "python": sys.version.split()[0], "torch": torch.__version__,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "seed": SEED})

# DATA 266 Lab 2 — Part 2: Fine-Tuning Stable Diffusion 1.5 with LoRA

Two controlled runs on the same data, split, base model, steps, batch order, noise/timestep sampling and evaluation; the **only** difference is LoRA rank/alpha (A: 4/4, B: 64/64).

Notebook order: run information (cell 1, required first) → **configuration (`MODE`, every parameter)** → reproducibility record → data → frozen prompts → fine-tuning → generation → automated evaluation → blinded human evaluation → report tables/figures → Hugging Face artifacts → safety.

Every number in the report tables is computed by this notebook from saved artifacts. Human ratings are never generated by code. Decisions are logged in `DECISIONS.md` (D-089 onwards for this implementation).

## 0. Configuration

Set `MODE`, then **Run All**:

| `MODE` | What Run All does | GPU | Typical time on RTX 5090 |
|---|---|---|---|
| `SETUP_ONLY` (saved default) | checks, dataset download/clean/split, creates or verifies the frozen `eval/p2_prompts.csv` and pins model revisions | no | minutes |
| `FULL_RUN` | everything: train A and B (resumable checkpoints), 120 images (resumable), KID/CLIP/LPIPS/memorization, final-adapter selection, blinded human-eval packet, report, HF staging | yes | see `DECISIONS.md` estimates |
| `REPORT_ONLY` | no training/generation/metric models: re-reads saved results, scores **real** ratings in `eval/p2_human_ratings.csv`, rebuilds tables, figures and model cards | no | seconds |

`FULL_RUN` refuses to train while `eval/` is uncommitted (`REQUIRE_COMMITTED_EVAL_FILES`, D-045). Completed runs, images and metrics are detected by fingerprint and skipped — they are never overwritten. Uploads need `UPLOAD_TO_HF = True` **and** the authorization phrase.

In [ ]:
# ===================== CONFIGURATION (single source of truth) =====================
import os, json
from pathlib import Path

# ---------------------------------------------------------------------------------------------------------------
# RUN MODE — normally the only value you change before "Run All".
#   "SETUP_ONLY"  : validate environment/config, download + clean + split the dataset, create/verify the frozen
#                   eval/p2_prompts.csv. No diffusion/metric model is loaded; nothing is trained or generated.
#   "FULL_RUN"    : SETUP_ONLY + train run A and run B (checkpointed, resumable) + generate the 120 images
#                   (resumable) + KID / CLIP / LPIPS / memorization + final-adapter selection + blinded
#                   human-evaluation materials + report tables/figures + Hugging Face staging folders.
#   "REPORT_ONLY" : no training, no generation, no metric models. Reuses saved outputs, scores REAL human ratings
#                   from eval/p2_human_ratings.csv (if present) and rebuilds tables, figures and model cards.
# ---------------------------------------------------------------------------------------------------------------
MODE = "SETUP_ONLY"
EXPERIMENT_ID = "P2-EXP-001"      # completed artifacts of an experiment ID are never overwritten (change the ID instead)

MODES = ("SETUP_ONLY", "FULL_RUN", "REPORT_ONLY")
if MODE not in MODES:
    raise ValueError(f"MODE must be one of {MODES}, got {MODE!r}")
DO_DATA_DOWNLOAD = MODE in ("SETUP_ONLY", "FULL_RUN")
DO_TRAIN = DO_GENERATE = DO_METRICS = DO_HUMAN_PREP = MODE == "FULL_RUN"
DO_REPORT = MODE in ("FULL_RUN", "REPORT_ONLY")

# --- Integrity gates (D-045: evaluation artifacts are committed before any training) ---
REQUIRE_COMMITTED_EVAL_FILES = True   # FULL_RUN stops before training unless eval/p2_prompts.csv is committed in Git
                                      # (set False only if the team consciously accepts an uncommitted prompt file)

# --- Hugging Face publication (OFF; requires explicit team authorization) ---
UPLOAD_TO_HF = False
HF_UPLOAD_AUTHORIZATION = ""          # must be exactly "UPLOAD AUTHORIZED BY TEAM" in addition to UPLOAD_TO_HF = True
HF_PRIVATE_REPOS = True               # repos are created private; making them public is a manual decision
HF_TOKEN_ENV = "HF_TOKEN"             # read from the environment only — never stored in the notebook/repo
HF_REPO_RUN_A = HF_REPO_RUN_B = HF_REPO_FINAL = HF_REPO_IMAGES = None   # OPEN: e.g. "<user>/data266-p2-lora-r4"

# --- Environment & paths ---
try:
    import torch as _torch
    CUDA_AVAILABLE = _torch.cuda.is_available()
except ImportError:
    CUDA_AVAILABLE = False
ON_KAGGLE = Path("/kaggle/input").exists()
DEVICE = "cuda:0"                 # one GPU only (team rule 3); also set CUDA_VISIBLE_DEVICES=0 in the environment
REPO_DIR = Path(os.environ.get("LAB2_REPO_DIR") or ("/kaggle/working" if ON_KAGGLE else Path.cwd().parent))
P2_WORK_DIR = Path(os.environ.get("P2_WORK_DIR",   # adapters, generated images, caches: OUTSIDE the repo (D-059)
                   "/kaggle/working/p2_work" if ON_KAGGLE else Path.home() / "data266_lab2_work" / "p2"))
EVAL_DIR = REPO_DIR / "eval"      # frozen evaluation artifacts (committed to Git, D-045)
P2_PROMPTS_PATH = EVAL_DIR / "p2_prompts.csv"
P2_SPLIT_MANIFEST_PATH = EVAL_DIR / "p2_split_manifest.json"     # split indices + cleaning report (no images/captions)
P2_EVAL_FREEZE_PATH = EVAL_DIR / "p2_eval_freeze.json"           # hashes + provenance of the frozen files
P2_HUMAN_RATINGS_PATH = EVAL_DIR / "p2_human_ratings.csv"        # REAL ratings only; filled by the raters (D-058)
P2_RUNS_DIR = P2_WORK_DIR / "runs" / EXPERIMENT_ID
P2_GEN_ROOT = P2_WORK_DIR / "generation" / EXPERIMENT_ID
P2_RESULTS_DIR = P2_WORK_DIR / "results" / EXPERIMENT_ID
P2_HUMAN_ROOT = P2_WORK_DIR / "human_eval" / EXPERIMENT_ID
P2_HF_STAGING = P2_WORK_DIR / "hf_staging" / EXPERIMENT_ID
P2_SELECTED_POINTER = P2_WORK_DIR / "SELECTED_FINAL_ADAPTER.json"  # read by Part 3 (local adapter before any upload)

# --- Dataset (D-047 to D-050, D-064) ---
DATASET_ID = "reach-vb/pokemon-blip-captions"        # DECIDED (D-064): 833 rows, image/text, license stated on the card
DATASET_REVISION = None                              # None -> resolved once, then frozen in eval/p2_split_manifest.json
DATASET_LICENSE = "cc-by-nc-sa-4.0"                  # as stated on the dataset card; re-verified at data preparation
DATASET_REJECTED_ID = "lambda/naruto-blip-captions"  # considered; rejected for EXP-001: no license stated on its card
IMAGE_COLUMN, CAPTION_COLUMN = "image", "text"
SPLIT_SEED, TEST_FRACTION = 266, 0.10                # rubric
MIN_VALID_IMAGES = 500                               # rubric: at least 500 captioned images after cleaning
MIN_IMAGE_SIDE = 256                                 # TUNE: smaller images are removed
NEAR_DUP_HAMMING_MAX = 4                             # TUNE: 64-bit dHash distance for near-duplicates
RESOLUTION = 512                                     # SD 1.5 resolution (rubric)
RESIZE_FILTER = "lanczos"                            # short side -> 512, then center crop
RANDOM_FLIP = False                                  # DECIDED: no augmentation

# --- Base model and method (D-051, D-052) ---
BASE_MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"
BASE_MODEL_REVISION = None                           # None -> resolved once, then frozen in the experiment manifest
BASE_MODEL_VARIANT = "fp16"
BASE_MODEL_LICENSE = "creativeml-openrail-m"
WEIGHT_DTYPE = "float16"                             # frozen weights on the GPU; LoRA parameters kept in fp32
LORA_TARGET_MODULES = ("to_q", "to_k", "to_v", "to_out.0")   # UNet attention projections only
LORA_DROPOUT, LORA_INIT = 0.0, "gaussian"

# --- Runs: rank/alpha is the ONLY difference between A and B (D-053) ---
RUNS = {"A": {"rank": 4, "alpha": 4}, "B": {"rank": 64, "alpha": 64}}
TRAIN = {                                            # shared by both runs (D-054)
    "learning_rate": 1e-4,                           # TUNE (never tuned from evaluation results)
    "lr_scheduler": "cosine", "warmup_fraction": 0.05,
    "max_train_steps": 1500,                         # approved budget (D-054)
    "effective_batch": 4, "micro_batch": 4,          # micro-batch = upper bound; the memory probe may lower it (accum. raised)
    "gradient_checkpointing": False,                 # the memory probe may switch it on if no micro-batch fits
    "adam_betas": (0.9, 0.999), "adam_weight_decay": 1e-2, "adam_epsilon": 1e-8, "max_grad_norm": 1.0,
    "latent_mode": "online",                         # "online" | "cached_moments" | "cached_mean" (D-055)
    "cache_text_embeddings": True,                   # DECIDED: text encoder is frozen
    "seed": 266,
}
LATENT_MODES = ("online", "cached_moments", "cached_mean")
MICRO_BATCH_CANDIDATES = (4, 2, 1)                   # probed in order; the effective batch never changes
CHECKPOINT_EVERY = 250                               # optimizer steps between resumable checkpoints
KEEP_CHECKPOINTS = 2                                 # older checkpoints are pruned; the final adapter is never pruned
MAX_NONFINITE_LOSS_STEPS = 10                        # abort (resumable) after this many consecutive non-finite losses

# --- GPU memory (single 16 GB GPU requirement, team rule 3) ---
VRAM_LIMIT_MB = 16384
ENFORCE_VRAM_CAP = True          # caps the PyTorch caching allocator of THIS process at VRAM_LIMIT_MB on larger GPUs.
                                 # Preliminary check only: it does NOT prove real T4/16 GB compatibility (CUDA context,
                                 # cuBLAS workspaces and kernel choice differ). Real 16 GB hardware still has to be tested.
VRAM_PROBE_MARGIN = 0.85         # a micro-batch is accepted only if its probe peak <= 85 % of VRAM_LIMIT_MB

# --- Generation (D-056) ---
GEN = {"scheduler": "DPMSolverMultistep", "steps": 30, "guidance_scale": 7.5, "height": RESOLUTION,
       "width": RESOLUTION, "seeds": (0, 1), "lora_scale": 1.0, "negative_prompt": None, "safety_checker": True}
N_PROMPTS = {"in_domain": 10, "out_of_domain": 5, "compositional": 5}   # rubric
N_GRID_PROMPTS = 8                                                    # rubric
GRID_PROMPT_IDS = ("P01", "P02", "P03", "P04", "P11", "P12", "P16", "P17")   # fixed before generation (4/2/2)
EXPECTED_IMAGES = len(GEN["seeds"]) * sum(N_PROMPTS.values()) * 3       # base, A, B -> 120

# Out-of-domain and compositional prompts: written by the team BEFORE any image was generated; never edited
# after results exist. In-domain prompts are drawn from held-out TEST captions (see section 1b).
OUT_OF_DOMAIN_PROMPTS = (
    "a red double-decker bus driving through a rainy city street at night",
    "a bowl of ramen with a boiled egg on a wooden table, food photography",
    "a lighthouse on a rocky cliff at sunset",
    "an astronaut riding a bicycle on the surface of the moon",
    "a portrait of an old fisherman with a grey beard",
)
COMPOSITIONAL_PROMPTS = (
    "a blue cartoon creature with orange wings standing next to a green cartoon turtle",
    "a small yellow cartoon creature sitting on top of a large red mushroom",
    "two cartoon creatures facing each other, one made of fire and one made of water",
    "a purple cartoon dragon holding a white umbrella in the rain",
    "a pink round cartoon creature to the left of a black cat-like cartoon creature",
)

# --- Evaluation (D-057, D-058) ---
CLIP_MODEL_ID = "openai/clip-vit-large-patch14"      # CANDIDATE (CLIP score + memorization embeddings)
CLIP_MODEL_REVISION = None                           # None -> resolved once, then frozen in the experiment manifest
LPIPS_NET = "alex"
KID_SUBSETS = 100
MEMORIZATION_N, MEMORIZATION_SHOW = 10, 3            # rubric
HUMAN_EVAL_PAIRS, HUMAN_EVAL_SEED, HUMAN_EVAL_MIN_RATERS = 25, 266, 3   # rubric
HUMAN_EVAL_CRITERIA = ("domain_style", "prompt_match")

# --- Final adapter selection (pre-registered BEFORE any result exists; Part 3 uses the selected adapter) ---
FINAL_RUN = None                 # None -> FINAL_RUN_RULE; "A"/"B" -> explicit team override (recorded as such)
FINAL_RUN_RULE = ("lower in-domain KID mean; if |KID_A - KID_B| <= max(KID_A std, KID_B std) the KID difference "
                  "is treated as noise and the higher in-domain CLIP score decides")

# --- Licenses / model-card declarations (D-064: adapter license is a team decision; see DECISIONS.md) ---
ADAPTER_LICENSE = "other"
ADAPTER_LICENSE_NAME = "openrail-m-and-cc-by-nc-sa-4.0"
ADAPTER_LICENSE_NOTE = ("Base model under CreativeML OpenRAIL-M (use restrictions apply). Trained on "
                        "reach-vb/pokemon-blip-captions (CC BY-NC-SA 4.0): attribution required, non-commercial "
                        "use only, derivatives under the same terms. Depicts copyrighted characters: research only.")

print({"mode": MODE, "experiment_id": EXPERIMENT_ID, "train": DO_TRAIN, "generate": DO_GENERATE,
       "metrics": DO_METRICS, "report": DO_REPORT, "upload_to_hf": UPLOAD_TO_HF, "cuda": CUDA_AVAILABLE,
       "dataset": DATASET_ID, "base_model": BASE_MODEL_ID, "runs": RUNS, "steps": TRAIN["max_train_steps"],
       "repo_dir": str(REPO_DIR), "work_dir": str(P2_WORK_DIR)})

In [ ]:
# ===== 0b. Reproducibility record, GPU controls, persistence helpers (D-060, D-089, D-090) =====
import datetime as _dt
import gc, hashlib, platform, shutil, subprocess, sys, time
import importlib.metadata as _md
import numpy as np
import torch
from IPython.display import display

def utc_now():
    return _dt.datetime.now(_dt.timezone.utc).isoformat(timespec="seconds")

def sha256_file(path, chunk=2**20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def sha256_json(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()

def write_json(path, obj):
    """Atomic write (temp file + rename) so an interruption never leaves a half-written record."""
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, indent=1, default=str))
    os.replace(tmp, path)
    return path

def read_json(path):
    return json.loads(Path(path).read_text())

def pkg_version(dist):
    try:
        return _md.version(dist)
    except _md.PackageNotFoundError:
        return None

def _git(*args):
    try:
        r = subprocess.run(["git", "-C", str(REPO_DIR), *args], capture_output=True, text=True, timeout=30)
    except (FileNotFoundError, subprocess.TimeoutExpired):
        return None
    return r

def git_state():
    """Commit + dirty working tree (reported, never changed by the notebook)."""
    head = _git("rev-parse", "HEAD")
    if head is None:
        return {"git_available": False}
    if head.returncode != 0:
        return {"git_available": True, "is_repo_with_commit": False, "detail": head.stderr.strip()[:200]}
    status = _git("status", "--porcelain")
    dirty = [l for l in (status.stdout.splitlines() if status else []) if l.strip()]
    return {"git_available": True, "is_repo_with_commit": True, "commit": head.stdout.strip(),
            "dirty": bool(dirty), "dirty_files": dirty[:50]}

def git_file_committed(path):
    """True when `path` is tracked and identical to HEAD (D-045 freeze check)."""
    rel = str(Path(path).resolve().relative_to(REPO_DIR.resolve()))
    tracked = _git("ls-files", "--error-unmatch", rel)
    if tracked is None or tracked.returncode != 0:
        return False
    same = _git("diff", "--quiet", "HEAD", "--", rel)
    return same is not None and same.returncode == 0

def notebook_sha(name):
    p = REPO_DIR / "notebooks" / name
    return sha256_file(p) if p.exists() else None      # hash of the SAVED notebook file

# ---- GPU visibility and the 16 GB budget ----
GPU_INFO = {"cuda": CUDA_AVAILABLE}
if CUDA_AVAILABLE:
    _props = torch.cuda.get_device_properties(0)
    GPU_INFO.update({"name": _props.name, "total_mb": round(_props.total_memory / 2**20),
                     "capability": list(torch.cuda.get_device_capability(0)), "visible_devices": torch.cuda.device_count(),
                     "CUDA_VISIBLE_DEVICES": os.environ.get("CUDA_VISIBLE_DEVICES"),
                     "PYTORCH_CUDA_ALLOC_CONF": os.environ.get("PYTORCH_CUDA_ALLOC_CONF"), "torch_cuda": torch.version.cuda})
    if torch.cuda.device_count() > 1:
        print(f"WARNING: {torch.cuda.device_count()} GPUs visible. Only {DEVICE} is used; set CUDA_VISIBLE_DEVICES=0.")
    GPU_INFO["vram_cap_applied"] = False
    if ENFORCE_VRAM_CAP and GPU_INFO["total_mb"] > VRAM_LIMIT_MB:
        torch.cuda.set_per_process_memory_fraction(VRAM_LIMIT_MB / GPU_INFO["total_mb"], 0)
        GPU_INFO["vram_cap_applied"] = True
        GPU_INFO["vram_cap_note"] = ("PyTorch allocator capped at VRAM_LIMIT_MB for preliminary 16 GB testing; "
                                     "this is not proof of real 16 GB / T4 compatibility.")
torch.backends.cudnn.benchmark = False      # stable kernel choice; bit-exactness across GPUs is NOT claimed (D-060)

def mb(b):
    return round(b / 2**20)

def gpu_mem(tag=""):
    if not CUDA_AVAILABLE:
        return {"tag": tag}
    torch.cuda.synchronize()
    free, total = torch.cuda.mem_get_info(0)
    return {"tag": tag, "allocated_mb": mb(torch.cuda.memory_allocated(0)), "reserved_mb": mb(torch.cuda.memory_reserved(0)),
            "peak_allocated_mb": mb(torch.cuda.max_memory_allocated(0)), "device_used_mb": mb(total - free)}

def release_gpu(reset_peak=False):
    gc.collect()
    if CUDA_AVAILABLE:
        torch.cuda.empty_cache()
        if reset_peak:
            torch.cuda.reset_peak_memory_stats(0)

def oom_help(stage):
    return (f"CUDA out of memory during {stage}. Nothing completed was overwritten; the last checkpoint is kept. "
            f"Options: lower MICRO_BATCH_CANDIDATES, set TRAIN['gradient_checkpointing']=True, close other GPU "
            f"programs, then re-run (training resumes from its last checkpoint).")

def require_cuda(stage):
    if not CUDA_AVAILABLE:
        raise RuntimeError(f"{stage} needs a CUDA GPU; none is visible to this kernel.")

PACKAGES = ["torch", "torchvision", "triton", "numpy", "pillow", "pandas", "datasets", "pyarrow", "huggingface-hub",
            "transformers", "diffusers", "peft", "accelerate", "safetensors", "torchmetrics", "torch-fidelity",
            "lpips", "scipy", "matplotlib"]
ENV_RECORD = {"python": sys.version.split()[0], "platform": f"{platform.system()} {platform.machine()}",
              "packages": {d: pkg_version(d) for d in PACKAGES}, "gpu": GPU_INFO, "git": git_state(),
              "notebook_sha256": notebook_sha("p2_finetune.ipynb"), "mode": MODE, "experiment_id": EXPERIMENT_ID,
              "started_at": utc_now(), "seed": SPLIT_SEED}
P2_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RUN_MANIFEST_PATH = P2_RESULTS_DIR / f"run_manifest_{_dt.datetime.now(_dt.timezone.utc):%Y%m%dT%H%M%SZ}_{MODE}.json"
write_json(RUN_MANIFEST_PATH, {"environment": ENV_RECORD, "status": "STARTED"})

_missing = [d for d in PACKAGES if ENV_RECORD["packages"][d] is None]
if _missing:
    raise RuntimeError(f"Missing packages {_missing}: use the Lab 2 Docker kernel (implementation/docker).")
if ENV_RECORD["git"].get("dirty"):
    print(f"NOTE: the Git working tree is dirty ({len(ENV_RECORD['git']['dirty_files'])} changed paths); recorded in the run manifest.")
if REPO_DIR.resolve() in P2_WORK_DIR.resolve().parents or P2_WORK_DIR.resolve() == REPO_DIR.resolve():
    raise RuntimeError("P2_WORK_DIR must be outside the repository (D-059).")
print(json.dumps({k: ENV_RECORD[k] for k in ("python", "gpu", "git", "notebook_sha256")}, indent=1, default=str))
print("run manifest:", RUN_MANIFEST_PATH)

## 1. Data preparation (Task 2.1)

Load (pinned dataset revision) → remove corrupted/small/caption-less/duplicate/near-duplicate images (counts reported) → deterministic 90/10 split of the **cleaned** rows with seed 266 → 512×512. Leakage check: no near-duplicate across train/test. The split is frozen in `eval/p2_split_manifest.json`; a later run must reproduce it exactly or it stops.

In [ ]:
# ===== 1. Data preparation helpers (D-048 to D-050) =====
import io
from PIL import Image, ImageOps
from torch.utils.data import DataLoader, Dataset

_RESAMPLE = {"lanczos": Image.Resampling.LANCZOS, "bicubic": Image.Resampling.BICUBIC}

def image_bytes(value):
    """Raw encoded bytes from a datasets Image(decode=False) value, raw bytes, or a PIL image."""
    if isinstance(value, dict):
        return value.get("bytes") or (Path(value["path"]).read_bytes() if value.get("path") else None)
    if isinstance(value, (bytes, bytearray)):
        return bytes(value)
    if isinstance(value, Image.Image):
        buf = io.BytesIO(); value.save(buf, format="PNG"); return buf.getvalue()
    return None

def decode_rgb(raw):
    """(RGB image, None) or (None, reason). Transparency is composited on white; EXIF orientation applied."""
    if not raw:
        return None, "missing_image"
    try:
        with Image.open(io.BytesIO(raw)) as probe:
            probe.verify()
        img = Image.open(io.BytesIO(raw)); img.load()
    except Exception:  # noqa: BLE001 — any decoder failure means a corrupted image
        return None, "decode_error"
    img = ImageOps.exif_transpose(img)
    if img.mode in ("RGBA", "LA") or (img.mode == "P" and "transparency" in img.info):
        rgba = img.convert("RGBA")
        img = Image.alpha_composite(Image.new("RGBA", rgba.size, (255, 255, 255, 255)), rgba)
    return img.convert("RGB"), None

def dhash64(img):
    """64-bit difference hash (perceptual) used for near-duplicate detection."""
    g = np.asarray(img.convert("L").resize((9, 8), Image.Resampling.LANCZOS), dtype=np.int16)
    return int("".join("1" if b else "0" for b in (g[:, 1:] > g[:, :-1]).flatten()), 2)

def clean_records(records, min_side=None, hamming_max=None):
    """records: iterable of {"idx", "image", "text"} in dataset order; the first occurrence of a duplicate is kept.
    Returns (kept, removed, report). Kept rows hold raw bytes (decoded lazily) to bound memory."""
    min_side = MIN_IMAGE_SIDE if min_side is None else min_side
    hamming_max = NEAR_DUP_HAMMING_MAX if hamming_max is None else hamming_max
    kept, removed, exact = [], [], {}
    for r in records:
        raw, text = image_bytes(r.get("image")), (r.get("text") or "").strip()
        img, reason = decode_rgb(raw)
        dup_of = None
        if reason is None and min(img.size) < min_side:
            reason = "too_small"
        if reason is None and not text:
            reason = "missing_caption"
        if reason is None:
            px = hashlib.sha256(img.tobytes() + repr(img.size).encode()).hexdigest()
            if px in exact:
                reason, dup_of = "exact_duplicate", exact[px]
        if reason is None:
            h = dhash64(img)
            near = next((k for k in kept if (k["dhash"] ^ h).bit_count() <= hamming_max), None)
            if near is not None:
                reason, dup_of = "near_duplicate", near["idx"]
        if reason is None:
            exact[px] = r["idx"]
            kept.append({"idx": r["idx"], "text": text, "raw": raw, "size": img.size, "dhash": h})
        else:
            removed.append({"idx": r["idx"], "reason": reason, "duplicate_of": dup_of})
    reasons = ["missing_image", "decode_error", "too_small", "missing_caption", "exact_duplicate", "near_duplicate"]
    report = {"total": len(kept) + len(removed), "kept": len(kept), "removed": len(removed),
              **{k: sum(x["reason"] == k for x in removed) for k in reasons}}
    return kept, removed, report

def split_train_test(kept, test_fraction=None, seed=None):
    """Deterministic 90/10 split of the CLEANED rows (dedup happens before the split; D-048, D-049)."""
    test_fraction = TEST_FRACTION if test_fraction is None else test_fraction
    seed = SPLIT_SEED if seed is None else seed
    ids = [k["idx"] for k in kept]
    perm = np.random.default_rng(seed).permutation(len(ids))
    n_test = int(round(test_fraction * len(ids)))
    return sorted(ids[i] for i in perm[n_test:]), sorted(ids[i] for i in perm[:n_test])

def preprocess(img, resolution=None):
    """Short side -> resolution (Lanczos), then center crop; no augmentation (D-050)."""
    resolution = RESOLUTION if resolution is None else resolution
    w, h = img.size
    s = resolution / min(w, h)
    img = img.resize((max(resolution, round(w * s)), max(resolution, round(h * s))), _RESAMPLE[RESIZE_FILTER])
    left, top = (img.width - resolution) // 2, (img.height - resolution) // 2
    return img.crop((left, top, left + resolution, top + resolution))

def to_model_tensor(img):
    """PIL RGB -> float32 CHW tensor in [-1, 1] (VAE input and LPIPS input convention)."""
    return torch.from_numpy(np.asarray(img, dtype=np.float32) / 127.5 - 1.0).permute(2, 0, 1).contiguous()

def tokenize_captions(tokenizer, captions):
    return tokenizer(list(captions), padding="max_length", max_length=tokenizer.model_max_length,
                     truncation=True, return_tensors="pt").input_ids

class CaptionedImages(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        return {"pixel_values": to_model_tensor(preprocess(decode_rgb(r["raw"])[0])), "text": r["text"], "idx": r["idx"]}

def collate_batch(batch):
    return {"pixel_values": torch.stack([b["pixel_values"] for b in batch]),
            "text": [b["text"] for b in batch], "idx": [b["idx"] for b in batch]}

def make_loader(rows, batch_size, seed, shuffle=True):
    """Seeded loader; num_workers=0 keeps the sample order reproducible across machines."""
    return DataLoader(CaptionedImages(rows), batch_size=batch_size, shuffle=shuffle, drop_last=True,
                      generator=torch.Generator().manual_seed(seed), collate_fn=collate_batch, num_workers=0)

def load_records(dataset_id=None, limit=None, revision=None):
    """Full dataset at a pinned revision: images left undecoded so corrupted files can be detected and counted."""
    from datasets import Image as HFImage, load_dataset
    ds = load_dataset(dataset_id or DATASET_ID, split="train", revision=revision).cast_column(IMAGE_COLUMN, HFImage(decode=False))
    for i in range(len(ds) if limit is None else min(limit, len(ds))):
        row = ds[i]
        yield {"idx": i, "image": row[IMAGE_COLUMN], "text": row[CAPTION_COLUMN]}

def resolve_revision(kind, repo_id, pinned=None):
    """Exact Hub commit for a dataset/model (pinned value wins)."""
    if pinned:
        return pinned
    from huggingface_hub import HfApi
    info = HfApi().dataset_info(repo_id) if kind == "dataset" else HfApi().model_info(repo_id)
    return info.sha

def stated_dataset_license(revision):
    from huggingface_hub import HfApi
    info = HfApi().dataset_info(DATASET_ID, revision=revision)
    meta = info.card_data.to_dict() if info.card_data else {}
    return meta.get("license") or next((t.split(":", 1)[1] for t in (info.tags or []) if t.startswith("license:")), None)

def pixels_uint8(rows):
    """Preprocessed 512x512 RGB uint8 array [N, H, W, 3] — exactly the training/evaluation preprocessing."""
    return np.stack([np.asarray(preprocess(decode_rgb(r["raw"])[0]), dtype=np.uint8) for r in rows])

In [ ]:
# ===== 1. Data preparation — RUN (all modes; REPORT_ONLY only re-reads the frozen manifest) =====
def content_fingerprint(kept):
    """Hash of the exact cleaned content: (dataset index, sha256 of the raw image bytes, caption)."""
    return sha256_json([(k["idx"], hashlib.sha256(k["raw"]).hexdigest(), k["text"]) for k in kept])

def min_cross_split_dhash(by_idx, train_ids, test_ids):
    tr = np.array([by_idx[i]["dhash"] for i in train_ids], dtype=np.uint64)
    best = 64
    for j in test_ids:
        x = np.bitwise_xor(tr, np.uint64(by_idx[j]["dhash"]))
        bits = np.unpackbits(x.view(np.uint8).reshape(-1, 8), axis=1).sum(axis=1)
        best = min(best, int(bits.min()))
    return best

def prepare_dataset():
    existing = read_json(P2_SPLIT_MANIFEST_PATH) if P2_SPLIT_MANIFEST_PATH.exists() else None
    if existing and DATASET_REVISION and existing["dataset_revision"] != DATASET_REVISION:
        raise RuntimeError(f"DATASET_REVISION {DATASET_REVISION} != frozen {existing['dataset_revision']}: "
                           "the split/prompts are frozen for this revision; use a new EXPERIMENT_ID and eval files.")
    revision = existing["dataset_revision"] if existing else resolve_revision("dataset", DATASET_ID, DATASET_REVISION)
    license_stated = stated_dataset_license(revision)
    if license_stated != DATASET_LICENSE:
        raise RuntimeError(f"Dataset card license {license_stated!r} != configured {DATASET_LICENSE!r}: review before use.")
    t0 = time.perf_counter()
    records = list(load_records(revision=revision))
    kept, removed, report = clean_records(records)
    del records
    if report["kept"] < MIN_VALID_IMAGES:
        raise RuntimeError(f"Only {report['kept']} valid images after cleaning (< {MIN_VALID_IMAGES}).")
    train_ids, test_ids = split_train_test(kept)
    by_idx = {k["idx"]: k for k in kept}
    if set(train_ids) & set(test_ids) or len(train_ids) + len(test_ids) != len(kept):
        raise RuntimeError("Split is not a disjoint partition of the cleaned rows.")
    cross = min_cross_split_dhash(by_idx, train_ids, test_ids)
    if cross <= NEAR_DUP_HAMMING_MAX:
        raise RuntimeError(f"Near-duplicate across train/test (dHash distance {cross}): evaluation leakage.")
    manifest = {"dataset_id": DATASET_ID, "dataset_revision": revision, "license_stated": license_stated,
                "cleaning": {"min_image_side": MIN_IMAGE_SIDE, "near_dup_hamming_max": NEAR_DUP_HAMMING_MAX},
                "split": {"seed": SPLIT_SEED, "test_fraction": TEST_FRACTION, "method": "numpy default_rng permutation of cleaned rows"},
                "report": report, "removed": removed, "train_ids": train_ids, "test_ids": test_ids,
                "min_train_test_dhash_distance": cross, "content_sha256": content_fingerprint(kept)}
    if existing:
        frozen_keys = ("dataset_revision", "cleaning", "split", "report", "train_ids", "test_ids", "content_sha256")
        diff = [k for k in frozen_keys if existing.get(k) != manifest[k]]
        if diff:
            raise RuntimeError(f"Re-derived split differs from the frozen eval/p2_split_manifest.json in {diff}. "
                               "Nothing was overwritten. Investigate (dataset changed?) before continuing.")
        manifest = existing
        print("Split reproduced exactly and matches the frozen manifest.")
    else:
        manifest["created_at"] = utc_now()
        write_json(P2_SPLIT_MANIFEST_PATH, manifest)
        print("Split manifest created:", P2_SPLIT_MANIFEST_PATH)
    train_rows, test_rows = [by_idx[i] for i in train_ids], [by_idx[i] for i in test_ids]
    words = [len(r["text"].split()) for r in kept]
    sizes = {}
    for r in kept:
        sizes[f"{r['size'][0]}x{r['size'][1]}"] = sizes.get(f"{r['size'][0]}x{r['size'][1]}", 0) + 1
    stats = {"dataset_id": DATASET_ID, "revision": revision, "license": license_stated, **report,
             "train": len(train_rows), "test": len(test_rows), "unique_captions": len({r["text"] for r in kept}),
             "caption_words_mean": round(float(np.mean(words)), 2), "caption_words_max": int(max(words)),
             "original_sizes_top5": sorted(sizes.items(), key=lambda kv: -kv[1])[:5],
             "min_train_test_dhash_distance": cross, "prepare_seconds": round(time.perf_counter() - t0, 1)}
    write_json(P2_RESULTS_DIR / "dataset_stats.json", stats)
    return train_rows, test_rows, manifest, stats

if DO_DATA_DOWNLOAD:
    TRAIN_ROWS, TEST_ROWS, SPLIT_MANIFEST, DATASET_STATS = prepare_dataset()
    DATASET_REVISION_RESOLVED = SPLIT_MANIFEST["dataset_revision"]
    print(json.dumps(DATASET_STATS, indent=1))
    preview = Image.new("RGB", (8 * 128, 2 * 128), "white")
    for i, r in enumerate(TRAIN_ROWS[:16]):
        preview.paste(preprocess(decode_rgb(r["raw"])[0]).resize((128, 128)), ((i % 8) * 128, (i // 8) * 128))
    (P2_RESULTS_DIR / "figures").mkdir(exist_ok=True)
    preview.save(P2_RESULTS_DIR / "figures" / "dataset_preview_train.png")
    display(preview)
else:   # REPORT_ONLY: the frozen manifest and saved statistics are the record
    if not P2_SPLIT_MANIFEST_PATH.exists():
        raise RuntimeError("REPORT_ONLY needs eval/p2_split_manifest.json from an earlier SETUP_ONLY/FULL_RUN.")
    SPLIT_MANIFEST = read_json(P2_SPLIT_MANIFEST_PATH)
    DATASET_REVISION_RESOLVED = SPLIT_MANIFEST["dataset_revision"]
    DATASET_STATS = read_json(P2_RESULTS_DIR / "dataset_stats.json") if (P2_RESULTS_DIR / "dataset_stats.json").exists() else {}
    TRAIN_ROWS = TEST_ROWS = None
    print("REPORT_ONLY: using the frozen split manifest;", {k: DATASET_STATS.get(k) for k in ("kept", "train", "test")})

### 1b. Frozen evaluation prompts (rubric 2.3.1)

20 prompts = 10 in-domain (held-out **test** captions, drawn with seed 266 among test rows with a unique caption) + 5 out-of-domain + 5 compositional (written before any generation). Created once, hashed in `eval/p2_eval_freeze.json`, then only verified. **Commit `eval/` before `FULL_RUN`.**

In [ ]:
# ===== 1b. Frozen evaluation prompts: eval/p2_prompts.csv (D-045, D-056, D-091) =====
import csv
import pandas as pd

PROMPT_COLUMNS = ["prompt_id", "group", "prompt", "source_idx"]

def draw_in_domain_prompts(test_rows, n=None, seed=None):
    """In-domain prompts are TEST-split captions drawn deterministically (rubric 2.3.1)."""
    n = N_PROMPTS["in_domain"] if n is None else n
    pick = np.random.default_rng(SPLIT_SEED if seed is None else seed).choice(len(test_rows), size=n, replace=False)
    return [(test_rows[i]["idx"], test_rows[i]["text"]) for i in sorted(pick)]

def unique_caption_rows(rows):
    """First row (dataset order) per distinct caption, so no two in-domain prompts are the same text."""
    seen, out = set(), []
    for r in sorted(rows, key=lambda r: r["idx"]):
        key = " ".join(r["text"].lower().split())
        if key not in seen:
            seen.add(key); out.append(r)
    return out

def build_prompt_table(in_domain, out_of_domain, compositional):
    rows = [("in_domain", p, src) for src, p in in_domain] + [("out_of_domain", p, None) for p in out_of_domain] \
        + [("compositional", p, None) for p in compositional]
    df = pd.DataFrame([{"prompt_id": f"P{i + 1:02d}", "group": g, "prompt": p, "source_idx": s}
                       for i, (g, p, s) in enumerate(rows)], columns=PROMPT_COLUMNS)
    df["source_idx"] = pd.Series([None if s is None else int(s) for _, _, s in rows], dtype=object)   # no NaN floats
    counts = df["group"].value_counts().to_dict()
    if counts != {k: v for k, v in N_PROMPTS.items() if v}:
        raise ValueError(f"prompt groups {counts} != required {N_PROMPTS}")
    return df

def write_prompts_csv(df, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f, lineterminator="\n")
        w.writerow(PROMPT_COLUMNS)
        for r in df.itertuples(index=False):
            w.writerow([r.prompt_id, r.group, r.prompt, "" if r.source_idx is None else int(r.source_idx)])

def read_prompts_csv(path):
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    df["source_idx"] = pd.Series([int(s) if s.strip() else None for s in df["source_idx"]], dtype=object)
    return df

def validate_prompts(df, manifest, test_rows=None, train_rows=None):
    problems = []
    if list(df.columns) != PROMPT_COLUMNS:
        problems.append(f"columns {list(df.columns)} != {PROMPT_COLUMNS}")
        return problems, {}
    if list(df["prompt_id"]) != [f"P{i + 1:02d}" for i in range(sum(N_PROMPTS.values()))]:
        problems.append("prompt_id must be P01..P20 in order")
    if df["group"].value_counts().to_dict() != N_PROMPTS:
        problems.append(f"group counts {df['group'].value_counts().to_dict()} != {N_PROMPTS}")
    ind = df[df["group"] == "in_domain"]
    if not set(ind["source_idx"]) <= set(manifest["test_ids"]):
        problems.append("an in-domain source_idx is not in the held-out TEST split")
    if df[df["group"] != "in_domain"]["source_idx"].map(lambda v: v is not None).any():
        problems.append("only in-domain prompts may carry a source_idx")
    if list(df[df["group"] == "out_of_domain"]["prompt"]) != list(OUT_OF_DOMAIN_PROMPTS):
        problems.append("out-of-domain prompts differ from the frozen configuration")
    if list(df[df["group"] == "compositional"]["prompt"]) != list(COMPOSITIONAL_PROMPTS):
        problems.append("compositional prompts differ from the frozen configuration")
    if df["prompt"].str.strip().eq("").any() or not df["prompt"].is_unique:
        problems.append("empty or duplicated prompt text")
    info = {}
    if test_rows is not None:
        cap = {r["idx"]: r["text"] for r in test_rows}
        bad = [r.prompt_id for r in ind.itertuples() if cap.get(r.source_idx) != r.prompt]
        if bad:
            problems.append(f"in-domain prompts {bad} do not equal the cleaned test caption at source_idx")
    if train_rows is not None:
        train_caps = {r["text"] for r in train_rows}
        info["in_domain_prompts_also_a_train_caption"] = int(sum(p in train_caps for p in ind["prompt"]))
    return problems, info

def freeze_prompts(test_rows, train_rows):
    if P2_PROMPTS_PATH.exists():
        df = read_prompts_csv(P2_PROMPTS_PATH)
        problems, info = validate_prompts(df, SPLIT_MANIFEST, test_rows, train_rows)
        if problems:
            raise RuntimeError(f"Frozen {P2_PROMPTS_PATH.name} failed validation (NOT overwritten): {problems}")
        freeze = read_json(P2_EVAL_FREEZE_PATH) if P2_EVAL_FREEZE_PATH.exists() else None
        sha = sha256_file(P2_PROMPTS_PATH)
        if freeze and freeze["p2_prompts_sha256"] != sha:
            raise RuntimeError("eval/p2_prompts.csv changed after it was frozen (hash mismatch). Restore it from Git.")
        if freeze is None:
            freeze = {"p2_prompts_sha256": sha, "note": "freeze record written after the prompt file already existed",
                      "recorded_at": utc_now()}
            write_json(P2_EVAL_FREEZE_PATH, freeze)
        print(f"Prompt file verified (sha256 {sha[:16]}…).")
        return df, freeze, info
    if test_rows is None:
        raise RuntimeError("eval/p2_prompts.csv is missing; run SETUP_ONLY or FULL_RUN first.")
    df = build_prompt_table(draw_in_domain_prompts(unique_caption_rows(test_rows)), OUT_OF_DOMAIN_PROMPTS, COMPOSITIONAL_PROMPTS)
    problems, info = validate_prompts(df, SPLIT_MANIFEST, test_rows, train_rows)
    if problems:
        raise RuntimeError(f"New prompt table failed validation: {problems}")
    write_prompts_csv(df, P2_PROMPTS_PATH)
    freeze = {"p2_prompts_sha256": sha256_file(P2_PROMPTS_PATH), "p2_split_manifest_sha256": sha256_file(P2_SPLIT_MANIFEST_PATH),
              "created_at": utc_now(), "dataset_id": DATASET_ID, "dataset_revision": SPLIT_MANIFEST["dataset_revision"],
              "in_domain_method": (f"numpy default_rng({SPLIT_SEED}).choice over cleaned TEST rows with a unique caption "
                                   "(dataset order); prompt = the cleaned test caption at source_idx"),
              "out_of_domain_and_compositional": "written by the team before any generation (configuration cell)",
              "model_results_used": False, "notebook_sha256": ENV_RECORD["notebook_sha256"], "git": ENV_RECORD["git"]}
    write_json(P2_EVAL_FREEZE_PATH, freeze)
    print("Created and froze", P2_PROMPTS_PATH, "— commit eval/ BEFORE training (D-045).")
    return df, freeze, info

PROMPTS_DF, EVAL_FREEZE, PROMPT_INFO = freeze_prompts(TEST_ROWS, TRAIN_ROWS)
PROMPTS_SHA256 = EVAL_FREEZE["p2_prompts_sha256"]
EVAL_COMMITTED = {p.name: git_file_committed(p) for p in (P2_PROMPTS_PATH, P2_SPLIT_MANIFEST_PATH, P2_EVAL_FREEZE_PATH)}
print({"prompts_sha256": PROMPTS_SHA256, "committed_in_git": EVAL_COMMITTED, **PROMPT_INFO})
display(PROMPTS_DF)

## 2. Fine-tuning (Task 2.2)

Run A (rank 4) and run B (rank 64): identical data, batch schedule, per-step noise/timestep RNG, optimizer, LR schedule, effective batch and micro-batch plan (memory probe on the larger run). fp16 frozen weights, fp32 LoRA, AMP + GradScaler, UNet in train mode, seeded LoRA init. Atomic checkpoints every `CHECKPOINT_EVERY` steps; an interrupted run resumes from its last verified checkpoint; a completed run is never retrained. Exported adapters carry full provenance (`adapter_metadata.json`).

In [ ]:
# ===== 2. Fine-tuning helpers (D-051 to D-055, D-092) =====
import contextlib
from safetensors.torch import load_file, save_file

def run_config(run):
    """Complete configuration of one run: the shared TRAIN settings plus the run's rank/alpha (the only difference)."""
    return {"run": run, **RUNS[run], **TRAIN, "steps": TRAIN["max_train_steps"],
            "grad_accum": TRAIN["effective_batch"] // TRAIN["micro_batch"], "base_model": BASE_MODEL_ID,
            "dataset": DATASET_ID, "target_modules": LORA_TARGET_MODULES, "lora_dropout": LORA_DROPOUT, "lora_init": LORA_INIT}

def torch_dtype(name):
    return getattr(torch, name)

def _variant_kw():
    return {"variant": BASE_MODEL_VARIANT} if BASE_MODEL_VARIANT else {}

def _model_kw():
    """Every SD 1.5 load uses the same frozen Hub revision (recorded in the experiment manifest)."""
    return {**_variant_kw(), "revision": BASE_REVISION_RESOLVED}

def load_sd_components(device=DEVICE, need=("tokenizer", "text_encoder", "vae", "unet", "noise_scheduler")):
    """Load only the SD 1.5 components a stage needs (sequential GPU residency, as in Part 1 D-037)."""
    from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel
    from transformers import CLIPTextModel, CLIPTokenizer
    dt, out, rev = torch_dtype(WEIGHT_DTYPE), {}, BASE_REVISION_RESOLVED
    if "tokenizer" in need:
        out["tokenizer"] = CLIPTokenizer.from_pretrained(BASE_MODEL_ID, subfolder="tokenizer", revision=rev)
    if "text_encoder" in need:
        out["text_encoder"] = CLIPTextModel.from_pretrained(BASE_MODEL_ID, subfolder="text_encoder", torch_dtype=dt,
                                                            **_model_kw()).to(device).requires_grad_(False).eval()
    if "vae" in need:
        out["vae"] = AutoencoderKL.from_pretrained(BASE_MODEL_ID, subfolder="vae", torch_dtype=dt,
                                                   **_model_kw()).to(device).requires_grad_(False).eval()
    if "unet" in need:
        out["unet"] = UNet2DConditionModel.from_pretrained(BASE_MODEL_ID, subfolder="unet", torch_dtype=dt,
                                                           **_model_kw()).to(device).requires_grad_(False)
    if "noise_scheduler" in need:
        out["noise_scheduler"] = DDPMScheduler.from_pretrained(BASE_MODEL_ID, subfolder="scheduler", revision=rev)
    return out

def add_lora(unet, rank, alpha, seed=None):
    """Attach a PEFT LoRA adapter to the UNet attention projections; trainable parameters are cast to fp32.
    The global RNG is seeded first, so the Gaussian LoRA initialisation is reproducible."""
    from peft import LoraConfig
    unet.requires_grad_(False)
    if seed is not None:
        torch.manual_seed(seed)
    unet.add_adapter(LoraConfig(r=rank, lora_alpha=alpha, lora_dropout=LORA_DROPOUT, init_lora_weights=LORA_INIT,
                                target_modules=list(LORA_TARGET_MODULES)))
    params = [p for p in unet.parameters() if p.requires_grad]
    for p in params:
        p.data = p.data.float()
    return params

def count_params(module):
    total = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return {"total": total, "trainable": trainable, "trainable_pct": round(100.0 * trainable / total, 4)}

@torch.no_grad()
def encode_text(tokenizer, text_encoder, captions):
    return text_encoder(tokenize_captions(tokenizer, captions).to(text_encoder.device))[0]

@torch.no_grad()
def vae_moments(vae, pixel_values):
    """Posterior mean and std of the VAE latent distribution (cacheable; D-055)."""
    dist = vae.encode(pixel_values.to(vae.device, vae.dtype)).latent_dist
    return dist.mean, dist.std

def latents_from_moments(mean, std, mode, generator, scaling_factor):
    """'online' and 'cached_moments' draw a FRESH posterior sample every time (same distribution as normal VAE
    training); 'cached_mean' is deterministic. No single stochastic sample is ever frozen (D-055)."""
    if mode not in LATENT_MODES:
        raise ValueError(f"unknown latent mode {mode!r}")
    z = mean if mode == "cached_mean" else \
        mean + std * torch.randn(mean.shape, generator=generator, device=mean.device, dtype=mean.dtype)
    return z * scaling_factor

def diffusion_loss(unet, noise_scheduler, latents, text_emb, generator):
    """Standard SD 1.5 epsilon-prediction MSE (as in diffusers train_text_to_image_lora.py)."""
    if noise_scheduler.config.prediction_type != "epsilon":
        raise ValueError(f"unexpected prediction_type {noise_scheduler.config.prediction_type!r}")
    noise = torch.randn(latents.shape, generator=generator, device=latents.device, dtype=latents.dtype)
    t = torch.randint(0, noise_scheduler.config.num_train_timesteps, (latents.shape[0],),
                      generator=generator, device=latents.device).long()
    pred = unet(noise_scheduler.add_noise(latents, noise, t), t, encoder_hidden_states=text_emb, return_dict=False)[0]
    return torch.nn.functional.mse_loss(pred.float(), noise.float())

def amp_tools(device):
    """fp16 autocast + GradScaler on CUDA; plain fp32 elsewhere (CPU plumbing tests)."""
    on_cuda = str(device).startswith("cuda") and WEIGHT_DTYPE == "float16"
    ctx = (lambda: torch.autocast("cuda", dtype=torch.float16)) if on_cuda else contextlib.nullcontext
    return ctx, torch.amp.GradScaler("cuda", enabled=on_cuda)

def make_optimizer(params, total_steps):
    from diffusers.optimization import get_scheduler
    opt = torch.optim.AdamW(params, lr=TRAIN["learning_rate"], betas=TRAIN["adam_betas"],
                            weight_decay=TRAIN["adam_weight_decay"], eps=TRAIN["adam_epsilon"])
    sched = get_scheduler(TRAIN["lr_scheduler"], optimizer=opt, num_training_steps=total_steps,
                          num_warmup_steps=int(TRAIN["warmup_fraction"] * total_steps))
    return opt, sched

def optimizer_step(unet, params, opt, sched, scaler, autocast, noise_scheduler, micro_batches, generator):
    """One optimizer step over gradient-accumulation micro-batches [(latents, text_emb), ...]."""
    losses = []
    for latents, emb in micro_batches:
        with autocast():
            loss = diffusion_loss(unet, noise_scheduler, latents, emb, generator)
        scaler.scale(loss / len(micro_batches)).backward()
        losses.append(float(loss.detach()))
    scaler.unscale_(opt)
    grad_norm = float(torch.nn.utils.clip_grad_norm_(params, TRAIN["max_grad_norm"]))
    scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)
    return float(np.mean(losses)), grad_norm

def save_lora(unet, out_dir):
    """Save the adapter in diffusers format (loadable with pipe.load_lora_weights; Part 3 reuses it)."""
    from diffusers import StableDiffusionPipeline
    from diffusers.utils import convert_state_dict_to_diffusers
    from peft.utils import get_peft_model_state_dict
    StableDiffusionPipeline.save_lora_weights(save_directory=str(out_dir), safe_serialization=True,
                                              unet_lora_layers=convert_state_dict_to_diffusers(get_peft_model_state_dict(unet)))

# ---------- deterministic, resumable schedule (identical for run A and run B) ----------
def training_schedule(n_rows, steps=None, eff=None, seed=None):
    """[steps, effective_batch] training-row positions. Epoch e is a fresh permutation from default_rng([seed, e]);
    the incomplete last batch of an epoch is dropped. Fully determined by (n_rows, steps, batch, seed), so both runs
    see the same batches in the same order and a resumed run continues exactly where it stopped."""
    steps = TRAIN["max_train_steps"] if steps is None else steps
    eff = TRAIN["effective_batch"] if eff is None else eff
    seed = TRAIN["seed"] if seed is None else seed
    usable, idx, epoch = (n_rows // eff) * eff, [], 0
    while len(idx) < steps * eff:
        idx.extend(np.random.default_rng([seed, epoch]).permutation(n_rows)[:usable].tolist()); epoch += 1
    return np.asarray(idx[:steps * eff], dtype=np.int64).reshape(steps, eff)

def step_generator(step, device=DEVICE):
    """Noise, timesteps and posterior samples of optimizer step `step` depend only on (seed, step):
    identical across runs and unaffected by interruption/resume."""
    return torch.Generator(device=device).manual_seed(TRAIN["seed"] * 1_000_003 + int(step))

def pixel_batch(pixels_u8, positions):
    """uint8 [N,H,W,3] rows -> float32 [-1, 1] NCHW on the GPU (same convention as to_model_tensor)."""
    x = torch.from_numpy(pixels_u8[np.asarray(positions)]).to(DEVICE, torch.float32)
    return x.permute(0, 3, 1, 2).contiguous() / 127.5 - 1.0

# ---------- memory probe (single 16 GB GPU) ----------
def probe_training_memory(pixels_u8, text_emb):
    """Largest micro-batch (then gradient checkpointing) whose forward+backward peak fits VRAM_PROBE_MARGIN of the
    16 GB budget, measured on the LARGER adapter (run B) so both runs can use the same plan. No weights are updated."""
    require_cuda("memory probe")
    eff = TRAIN["effective_batch"]
    micro_opts = [m for m in MICRO_BATCH_CANDIDATES if eff % m == 0 and m <= TRAIN["micro_batch"]]
    gc_opts = [TRAIN["gradient_checkpointing"]] + ([True] if not TRAIN["gradient_checkpointing"] else [])
    big = max(RUNS, key=lambda r: RUNS[r]["rank"])
    trials = []
    for use_gc in gc_opts:
        for micro in micro_opts:
            comps = unet = None
            release_gpu(reset_peak=True)
            try:
                comps = load_sd_components(need=("vae", "unet", "noise_scheduler"))
                unet = comps["unet"]
                if use_gc:
                    unet.enable_gradient_checkpointing()
                add_lora(unet, RUNS[big]["rank"], RUNS[big]["alpha"], seed=TRAIN["seed"])
                unet.train()
                autocast, scaler = amp_tools(DEVICE)
                g = torch.Generator(device=DEVICE).manual_seed(0)
                mean, std = vae_moments(comps["vae"], pixel_batch(pixels_u8, range(micro)))
                lat = latents_from_moments(mean, std, "online", g, comps["vae"].config.scaling_factor)
                with autocast():
                    loss = diffusion_loss(unet, comps["noise_scheduler"], lat, text_emb[:micro].to(DEVICE), g)
                scaler.scale(loss).backward()
                peak = gpu_mem("probe")["peak_allocated_mb"]
                ok = peak <= VRAM_PROBE_MARGIN * VRAM_LIMIT_MB
                trials.append({"micro_batch": micro, "gradient_checkpointing": use_gc, "peak_mb": peak, "fits": ok})
            except torch.cuda.OutOfMemoryError:
                trials.append({"micro_batch": micro, "gradient_checkpointing": use_gc, "peak_mb": None, "fits": False, "oom": True})
            finally:
                comps = unet = None
                release_gpu()
            if trials[-1]["fits"]:
                return {"micro_batch": micro, "grad_accum": eff // micro, "gradient_checkpointing": use_gc,
                        "probe_run": big, "trials": trials, "vram_limit_mb": VRAM_LIMIT_MB, "margin": VRAM_PROBE_MARGIN}
    raise RuntimeError(f"No micro-batch/checkpointing option fits {VRAM_PROBE_MARGIN:.0%} of {VRAM_LIMIT_MB} MB: {trials}")

# ---------- checkpoints: atomic save, validated resume ----------
def _ckpt_dirs(run_dir):
    return sorted((run_dir / "checkpoints").glob("step_?????"), key=lambda p: int(p.name.split("_")[1]))

def save_checkpoint(run_dir, step, unet, opt, sched, scaler, history, elapsed_s, peak_mb, fingerprint, sessions=None):
    from peft.utils import get_peft_model_state_dict
    final = run_dir / "checkpoints" / f"step_{step:05d}"
    tmp = final.with_name(final.name + ".tmp")
    shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir(parents=True)
    save_file({k: v.detach().cpu().contiguous() for k, v in get_peft_model_state_dict(unet).items()}, str(tmp / "lora.safetensors"))
    torch.save({"optimizer": opt.state_dict(), "scheduler": sched.state_dict(), "scaler": scaler.state_dict()},
               tmp / "training_state.pt")
    write_json(tmp / "history.json", history)
    write_json(tmp / "checkpoint.json", {"step": step, "fingerprint": fingerprint, "elapsed_s": elapsed_s, "peak_mb": peak_mb,
                                         "sessions": sessions or [], "created_at": utc_now(), "lora_sha256": sha256_file(tmp / "lora.safetensors"),
                                         "state_sha256": sha256_file(tmp / "training_state.pt")})
    if final.exists():
        shutil.rmtree(final)
    os.replace(tmp, final)
    for old in _ckpt_dirs(run_dir)[:-KEEP_CHECKPOINTS]:
        shutil.rmtree(old, ignore_errors=True)
    return final

def latest_valid_checkpoint(run_dir, fingerprint):
    """Newest checkpoint whose metadata, fingerprint and file hashes all verify; invalid ones are reported, kept."""
    for d in reversed(_ckpt_dirs(run_dir)):
        try:
            meta = read_json(d / "checkpoint.json")
            if meta["fingerprint"] != fingerprint:
                print(f"  checkpoint {d.name}: different configuration fingerprint — ignored (not deleted)"); continue
            if sha256_file(d / "lora.safetensors") != meta["lora_sha256"] or sha256_file(d / "training_state.pt") != meta["state_sha256"]:
                print(f"  checkpoint {d.name}: file hash mismatch (corrupt) — ignored (not deleted)"); continue
            return d, meta
        except Exception as e:  # noqa: BLE001
            print(f"  checkpoint {d.name}: unreadable ({type(e).__name__}) — ignored (not deleted)")
    return None, None

def load_checkpoint(d, unet, opt, sched, scaler):
    from peft.utils import set_peft_model_state_dict
    res = set_peft_model_state_dict(unet, load_file(str(d / "lora.safetensors")))
    if getattr(res, "unexpected_keys", None):
        raise RuntimeError(f"checkpoint {d} has unexpected LoRA keys: {res.unexpected_keys[:5]}")
    state = torch.load(d / "training_state.pt", map_location="cpu", weights_only=False)   # our own file
    opt.load_state_dict(state["optimizer"]); sched.load_state_dict(state["scheduler"]); scaler.load_state_dict(state["scaler"])
    return read_json(d / "history.json")

In [ ]:
# ===== 2a. Experiment manifest (frozen revisions) + pre-run self-checks (no model weights) =====
EXPERIMENT_MANIFEST_PATH = P2_RUNS_DIR / "experiment_manifest.json"

def experiment_manifest():
    """Created once per EXPERIMENT_ID (SETUP_ONLY or FULL_RUN) and then only verified: the Hub revisions of every
    model/dataset, the frozen evaluation hashes and the shared training settings."""
    if EXPERIMENT_MANIFEST_PATH.exists():
        m = read_json(EXPERIMENT_MANIFEST_PATH)
        checks = {"prompts_sha256": PROMPTS_SHA256, "split_manifest_sha256": sha256_file(P2_SPLIT_MANIFEST_PATH),
                  "dataset_revision": DATASET_REVISION_RESOLVED, "train": json.loads(json.dumps(TRAIN)), "runs": RUNS,
                  "gen": json.loads(json.dumps(GEN))}
        bad = [k for k, v in checks.items() if m.get(k) != v]
        for k, pinned in (("base_model_revision", BASE_MODEL_REVISION), ("clip_model_revision", CLIP_MODEL_REVISION)):
            if pinned and m.get(k) != pinned:
                bad.append(k)
        if bad:
            raise RuntimeError(f"{EXPERIMENT_MANIFEST_PATH} differs from the current configuration in {bad}. "
                               "Completed experiments are never silently changed: restore the settings or use a new EXPERIMENT_ID.")
        return m
    if MODE == "REPORT_ONLY":
        raise RuntimeError("REPORT_ONLY needs an existing experiment manifest (run FULL_RUN first).")
    m = {"experiment_id": EXPERIMENT_ID, "created_at": utc_now(),
         "base_model_id": BASE_MODEL_ID, "base_model_revision": resolve_revision("model", BASE_MODEL_ID, BASE_MODEL_REVISION),
         "base_model_variant": BASE_MODEL_VARIANT, "clip_model_id": CLIP_MODEL_ID,
         "clip_model_revision": resolve_revision("model", CLIP_MODEL_ID, CLIP_MODEL_REVISION),
         "dataset_id": DATASET_ID, "dataset_revision": DATASET_REVISION_RESOLVED,
         "prompts_sha256": PROMPTS_SHA256, "split_manifest_sha256": sha256_file(P2_SPLIT_MANIFEST_PATH),
         "train": json.loads(json.dumps(TRAIN)), "runs": RUNS, "gen": json.loads(json.dumps(GEN)),
         "final_run_rule": FINAL_RUN_RULE, "environment": ENV_RECORD}
    write_json(EXPERIMENT_MANIFEST_PATH, m)
    print("Experiment manifest created:", EXPERIMENT_MANIFEST_PATH)
    return m

EXP_MANIFEST = experiment_manifest()
BASE_REVISION_RESOLVED = EXP_MANIFEST["base_model_revision"]
CLIP_REVISION_RESOLVED = EXP_MANIFEST["clip_model_revision"]
print({k: EXP_MANIFEST[k] for k in ("base_model_revision", "clip_model_revision", "dataset_revision", "prompts_sha256")})

# ---- self-checks: cheap, CPU-only, run in every mode; any FAIL stops Run All before expensive work ----
def _synthetic_records():
    """24 in-memory rows: 18 distinct images + 6 planted problems (kept from the validated smoke test P2-L1)."""
    from PIL import ImageDraw
    rng = np.random.default_rng(266)
    def scene(size):
        img = Image.new("RGB", size, tuple(int(c) for c in rng.integers(0, 256, 3)))
        d = ImageDraw.Draw(img)
        for _ in range(6):
            x0, y0 = int(rng.integers(0, size[0] // 2)), int(rng.integers(0, size[1] // 2))
            box = [x0, y0, x0 + int(rng.integers(size[0] // 4, size[0] // 2)), y0 + int(rng.integers(size[1] // 4, size[1] // 2))]
            (d.ellipse if rng.random() < 0.5 else d.rectangle)(box, fill=tuple(int(c) for c in rng.integers(0, 256, 3)))
        return img
    def enc(img, fmt="PNG", **kw):
        buf = io.BytesIO(); img.save(buf, format=fmt, **kw); return buf.getvalue()
    sizes = [(600, 500), (500, 800), (1024, 1024), (640, 360), (300, 700), (768, 512)]
    base = [scene(sizes[i % len(sizes)]) for i in range(18)]
    rgba = base[15].convert("RGBA"); rgba.putalpha(Image.new("L", rgba.size, 128)); base[15] = rgba
    base[16], base[17] = base[16].convert("L"), base[17].convert("P")
    recs = [{"idx": i, "image": {"bytes": enc(im), "path": None}, "text": f"synthetic caption {i}"} for i, im in enumerate(base)]
    recs += [{"idx": 18, "image": {"bytes": enc(base[0])[:200], "path": None}, "text": "truncated png"},
             {"idx": 19, "image": {"bytes": b"definitely not an image", "path": None}, "text": "not an image"},
             {"idx": 20, "image": {"bytes": enc(base[0]), "path": None}, "text": "exact duplicate of 0"},
             {"idx": 21, "image": {"bytes": enc(base[1], "JPEG", quality=85), "path": None}, "text": "near duplicate of 1"},
             {"idx": 22, "image": {"bytes": enc(scene((600, 600))), "path": None}, "text": "   "},
             {"idx": 23, "image": {"bytes": enc(scene((120, 100))), "path": None}, "text": "too small"}]
    return recs

def self_checks():
    out = {}
    _, removed, report = clean_records(_synthetic_records())
    out["cleaning_removes_exactly_the_planted_problems"] = report == {
        "total": 24, "kept": 18, "removed": 6, "missing_image": 0, "decode_error": 2, "too_small": 1,
        "missing_caption": 1, "exact_duplicate": 1, "near_duplicate": 1}
    a, b = run_config("A"), run_config("B")
    out["runs_differ_only_in_rank_alpha"] = sorted(k for k in set(a) | set(b) if a.get(k) != b.get(k)) == ["alpha", "rank", "run"]
    s1, s2 = training_schedule(750), training_schedule(750)
    out["schedule_deterministic_and_complete"] = bool(np.array_equal(s1, s2)) and s1.shape == (TRAIN["max_train_steps"], TRAIN["effective_batch"])
    out["schedule_epochs_without_repeats"] = all(len(set(s1[i * 187:(i + 1) * 187].ravel())) == 748 for i in range(2))
    out["effective_batch_divisible"] = all(TRAIN["effective_batch"] % m == 0 for m in MICRO_BATCH_CANDIDATES)
    out["latent_mode_valid"] = TRAIN["latent_mode"] in LATENT_MODES
    out["grid_prompts_valid"] = len(GRID_PROMPT_IDS) == N_GRID_PROMPTS and set(GRID_PROMPT_IDS) <= set(PROMPTS_DF["prompt_id"])
    out["no_hf_token_in_notebook_globals"] = not any(isinstance(v, str) and v.startswith("hf_") and len(v) > 20
                                                     for v in globals().values())
    out["upload_disabled_unless_authorized"] = (not UPLOAD_TO_HF) or HF_UPLOAD_AUTHORIZATION == "UPLOAD AUTHORIZED BY TEAM"
    out["prompt_file_frozen_and_valid"] = P2_PROMPTS_PATH.exists() and sha256_file(P2_PROMPTS_PATH) == PROMPTS_SHA256
    return out

SELF_CHECKS = self_checks()
for k, v in SELF_CHECKS.items():
    print(f"CHECK {k}: {'PASS' if v else 'FAIL'}")
if not all(SELF_CHECKS.values()):
    raise RuntimeError(f"Self-checks failed: {[k for k, v in SELF_CHECKS.items() if not v]}")
if DO_TRAIN:
    require_cuda("FULL_RUN")
    not_committed = [k for k, v in EVAL_COMMITTED.items() if not v]
    if REQUIRE_COMMITTED_EVAL_FILES and not_committed:
        raise RuntimeError(f"FULL_RUN stopped BEFORE training: {not_committed} are not committed in Git (D-045). "
                           "Commit eval/ (see README 'Freeze the pre-experiment state'), then re-run. "
                           "Nothing was trained.")
print("Self-checks passed. Mode:", MODE)

In [ ]:
# ===== 2b. Fine-tuning — RUN (FULL_RUN trains; REPORT_ONLY only reads the completed records) =====
TRAINING_PLAN_PATH = P2_RUNS_DIR / "training_plan.json"

def run_fingerprint(run, plan, schedule_sha):
    return sha256_json({"experiment_id": EXPERIMENT_ID, "config": run_config(run), "base_model_revision": BASE_REVISION_RESOLVED,
                        "dataset_revision": DATASET_REVISION_RESOLVED, "split_sha256": EXP_MANIFEST["split_manifest_sha256"],
                        "micro_batch": plan["micro_batch"], "gradient_checkpointing": plan["gradient_checkpointing"],
                        "schedule_sha256": schedule_sha, "resolution": RESOLUTION})

def completed_run(run):
    """Validated record of a completed run, or None. A completed run is never retrained or overwritten."""
    done = P2_RUNS_DIR / run / "final" / "COMPLETE.json"
    if not done.exists():
        return None
    meta = read_json(done)
    w = P2_RUNS_DIR / run / "final" / "pytorch_lora_weights.safetensors"
    if not w.exists() or sha256_file(w) != meta["adapter_sha256"]:
        raise RuntimeError(f"Run {run}: COMPLETE.json exists but the adapter file is missing or its hash differs. "
                           "Not overwritten — investigate the work directory.")
    return meta

def train_run(run, pixels_u8, text_emb_cpu, schedule, plan):
    run_dir = P2_RUNS_DIR / run
    schedule_sha = hashlib.sha256(schedule.tobytes()).hexdigest()
    fingerprint = run_fingerprint(run, plan, schedule_sha)
    meta = completed_run(run)
    if meta is not None:
        if meta["fingerprint"] != fingerprint:
            raise RuntimeError(f"Run {run} was completed with a DIFFERENT configuration (fingerprint mismatch). "
                               "It is not overwritten: use a new EXPERIMENT_ID to train again.")
        print(f"Run {run}: already complete and valid — skipped (no retraining).")
        return meta
    final_dir = run_dir / "final"
    if final_dir.exists():   # an interrupted export: keep it aside, re-export from the last checkpoint
        os.replace(final_dir, run_dir / f"final_incomplete_{int(time.time())}")
    cfg, steps, eff, micro = run_config(run), TRAIN["max_train_steps"], TRAIN["effective_batch"], plan["micro_batch"]
    release_gpu(reset_peak=True)
    comps = load_sd_components(need=("vae", "unet", "noise_scheduler"))
    vae, unet, ns = comps["vae"], comps["unet"], comps["noise_scheduler"]
    del comps
    if plan["gradient_checkpointing"]:
        unet.enable_gradient_checkpointing()
    params = add_lora(unet, cfg["rank"], cfg["alpha"], seed=TRAIN["seed"])
    unet.train()                                         # train mode (dropout 0, but correct for any module)
    sizes = count_params(unet)
    opt, sched = make_optimizer(params, steps)
    autocast, scaler = amp_tools(DEVICE)
    history, start, elapsed_prev, peak_prev, sessions = [], 0, 0.0, 0, []
    ckpt_dir, ckpt = latest_valid_checkpoint(run_dir, fingerprint)
    if ckpt_dir is not None:
        history = load_checkpoint(ckpt_dir, unet, opt, sched, scaler)
        start, elapsed_prev, peak_prev = ckpt["step"], ckpt["elapsed_s"], ckpt["peak_mb"]
        sessions = ckpt.get("sessions", [])
        print(f"Run {run}: resuming from checkpoint step {start} (elapsed {elapsed_prev:.0f} s).")
    cache = None
    if TRAIN["latent_mode"] != "online":                 # cached_* modes: posterior moments computed once per session
        cache = [vae_moments(vae, pixel_batch(pixels_u8, range(i, min(i + 16, len(pixels_u8))))) for i in range(0, len(pixels_u8), 16)]
        cache = (torch.cat([c[0] for c in cache]), torch.cat([c[1] for c in cache]))
    sf, nonfinite, t0 = vae.config.scaling_factor, 0, time.perf_counter()
    torch.cuda.reset_peak_memory_stats(0)
    print(f"Run {run}: rank {cfg['rank']} / alpha {cfg['alpha']} | trainable {sizes['trainable']:,} ({sizes['trainable_pct']} %) "
          f"| steps {start}->{steps} | micro {micro} x accum {eff // micro} | GC {plan['gradient_checkpointing']}")
    try:
        for step in range(start, steps):
            g = step_generator(step)
            mbs = []
            for k in range(0, eff, micro):
                pos = schedule[step, k:k + micro]
                if cache is None:
                    mean, std = vae_moments(vae, pixel_batch(pixels_u8, pos))
                else:
                    sel = torch.as_tensor(pos, device=DEVICE)
                    mean, std = cache[0][sel], cache[1][sel]
                mbs.append((latents_from_moments(mean, std, TRAIN["latent_mode"], g, sf),
                            text_emb_cpu[torch.as_tensor(pos)].to(DEVICE, non_blocking=True)))
            lr = opt.param_groups[0]["lr"]
            loss, gnorm = optimizer_step(unet, params, opt, sched, scaler, autocast, ns, mbs, g)
            nonfinite = nonfinite + 1 if not np.isfinite(loss) else 0
            if nonfinite >= MAX_NONFINITE_LOSS_STEPS:
                raise RuntimeError(f"{nonfinite} consecutive non-finite losses at step {step + 1}; resumable from the last checkpoint.")
            history.append({"step": step + 1, "loss": loss, "grad_norm": gnorm, "lr": lr, "grad_scale": float(scaler.get_scale())})
            if (step + 1) % 50 == 0 or step + 1 == steps:
                recent = [h["loss"] for h in history[-50:]]
                print(f"  {run} step {step + 1:5d}/{steps}  loss(50-avg) {np.mean(recent):.4f}  lr {lr:.2e}  "
                      f"{(time.perf_counter() - t0) / max(1, step + 1 - start):.3f} s/step")
            if (step + 1) % CHECKPOINT_EVERY == 0 or step + 1 == steps:
                now_s = time.perf_counter() - t0
                pk = max(peak_prev, mb(torch.cuda.max_memory_allocated(0)))
                sess = sessions + [{"start_step": start, "end_step": step + 1, "seconds": round(now_s, 1), "gpu": GPU_INFO.get("name")}]
                save_checkpoint(run_dir, step + 1, unet, opt, sched, scaler, history, elapsed_prev + now_s, pk, fingerprint, sess)
    except torch.cuda.OutOfMemoryError as e:
        raise RuntimeError(oom_help(f"training run {run}")) from e
    session_s = time.perf_counter() - t0
    train_s = elapsed_prev + session_s
    peak = max(peak_prev, mb(torch.cuda.max_memory_allocated(0)))
    sessions = sessions + [{"start_step": start, "end_step": steps, "seconds": round(session_s, 1), "gpu": GPU_INFO.get("name")}]
    if not all(torch.isfinite(p).all() for p in params):
        raise RuntimeError(f"Run {run}: non-finite LoRA weights after training — not exported.")
    tmp = run_dir / "final.tmp"
    shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir(parents=True)
    save_lora(unet, tmp)
    wfile = tmp / "pytorch_lora_weights.safetensors"
    n_saved = len(load_file(str(wfile)))
    if n_saved != len(params):
        raise RuntimeError(f"Run {run}: exported {n_saved} tensors but trained {len(params)} LoRA parameters.")
    write_json(tmp / "loss_history.json", history)
    meta = {"experiment_id": EXPERIMENT_ID, "run": run, "rank": cfg["rank"], "alpha": cfg["alpha"], "fingerprint": fingerprint,
            "adapter_file": "pytorch_lora_weights.safetensors", "adapter_sha256": sha256_file(wfile), "adapter_tensors": n_saved,
            "format": "diffusers LoRA (StableDiffusionPipeline.load_lora_weights)",
            "base_model_id": BASE_MODEL_ID, "base_model_revision": BASE_REVISION_RESOLVED, "base_model_variant": BASE_MODEL_VARIANT,
            "dataset_id": DATASET_ID, "dataset_revision": DATASET_REVISION_RESOLVED, "dataset_license": DATASET_LICENSE,
            "split_manifest_sha256": EXP_MANIFEST["split_manifest_sha256"], "prompts_sha256": PROMPTS_SHA256,
            "train_rows": int(len(pixels_u8)), "config": cfg, "plan": {k: plan[k] for k in ("micro_batch", "grad_accum", "gradient_checkpointing")},
            "schedule_sha256": schedule_sha, "params": sizes, "steps": steps, "final_loss_avg50": float(np.mean([h["loss"] for h in history[-50:]])),
            "train_time_s": round(train_s, 1), "gpu_hours": round(train_s / 3600, 4), "peak_gpu_mb": peak, "sessions": sessions,
            "vram_cap_applied": GPU_INFO.get("vram_cap_applied"), "gpu": GPU_INFO.get("name"),
            "packages": {d: ENV_RECORD["packages"][d] for d in ("torch", "diffusers", "peft", "transformers")},
            "git": ENV_RECORD["git"], "notebook_sha256": ENV_RECORD["notebook_sha256"], "completed_at": utc_now()}
    write_json(tmp / "adapter_metadata.json", meta)
    os.replace(tmp, final_dir)
    write_json(final_dir / "COMPLETE.json", meta)        # written LAST: marks a valid, complete run
    del unet, vae, ns, params, opt, sched, cache
    release_gpu()
    print(f"Run {run}: complete — {train_s / 60:.1f} min, peak {peak} MB, adapter sha256 {meta['adapter_sha256'][:16]}…")
    return meta

RUN_META = {}
if DO_TRAIN:
    SCHEDULE = training_schedule(len(TRAIN_ROWS))
    PENDING_RUNS = [r for r in RUNS if completed_run(r) is None]
    TRAIN_PIXELS = TEXT_EMB = None
    if PENDING_RUNS:
        print(f"Runs to train/resume: {PENDING_RUNS}. Preparing tensors (preprocessed 512x512 uint8, cached text embeddings)…")
        TRAIN_PIXELS = pixels_uint8(TRAIN_ROWS)
        release_gpu(reset_peak=True)
        _c = load_sd_components(need=("tokenizer", "text_encoder"))
        TEXT_EMB = torch.cat([encode_text(_c["tokenizer"], _c["text_encoder"], [r["text"] for r in TRAIN_ROWS[i:i + 64]]).cpu()
                              for i in range(0, len(TRAIN_ROWS), 64)])
        del _c; release_gpu()                            # text encoder freed: embeddings cached (D-055)
    if TRAINING_PLAN_PATH.exists():
        TRAINING_PLAN = read_json(TRAINING_PLAN_PATH)    # frozen: a resumed/second run uses the same plan
        print("Training plan (frozen):", {k: TRAINING_PLAN[k] for k in ("micro_batch", "grad_accum", "gradient_checkpointing")})
    elif not PENDING_RUNS:
        raise RuntimeError("Runs are complete but training_plan.json is missing — work directory inconsistent.")
    else:
        TRAINING_PLAN = probe_training_memory(TRAIN_PIXELS, TEXT_EMB)
        TRAINING_PLAN["schedule_sha256"] = hashlib.sha256(SCHEDULE.tobytes()).hexdigest()
        TRAINING_PLAN["created_at"] = utc_now()
        write_json(TRAINING_PLAN_PATH, TRAINING_PLAN)
        print("Training plan (memory probe):", json.dumps(TRAINING_PLAN, indent=1))
    if TRAINING_PLAN["schedule_sha256"] != hashlib.sha256(SCHEDULE.tobytes()).hexdigest():
        raise RuntimeError("Batch schedule differs from the frozen training plan — refusing to train.")
    for _run in RUNS:                                    # A then B: identical data, schedule, RNG, optimizer, plan
        RUN_META[_run] = train_run(_run, TRAIN_PIXELS, TEXT_EMB, SCHEDULE, TRAINING_PLAN)
    del TEXT_EMB
else:
    for _run in RUNS:
        m = completed_run(_run)
        if m is not None:
            RUN_META[_run] = m
    print("Training not run in this mode. Completed runs found:", sorted(RUN_META) or "none")
TRAINING_COMPLETE = set(RUN_META) == set(RUNS)
if DO_REPORT and not TRAINING_COMPLETE:
    raise RuntimeError(f"Runs {sorted(set(RUNS) - set(RUN_META))} are not complete: run MODE='FULL_RUN' (it resumes).")

## 3. Image generation (Task 2.3)

20 frozen prompts × seeds 0/1 × base, A, B with identical sampler settings = **120 images**, each with a JSON sidecar (prompt, seed, model/adapter hash, safety-checker flag, settings). Output directory = generation fingerprint, so images from different adapters/settings are never mixed. Eight side-by-side grids (`GRID_PROMPT_IDS`).

In [ ]:
# ===== 3. Image generation: helpers + RUN (D-056, D-093) =====
from PIL import ImageDraw

MODELS = ("base", "A", "B")
IMAGE_METADATA_FIELDS = ("model", "prompt_id", "group", "prompt", "seed", "steps", "guidance_scale", "scheduler",
                         "lora_scale", "nsfw_flagged")

def image_metadata(model, prompt_row, seed, nsfw_flagged):
    rec = {"model": model, "prompt_id": prompt_row["prompt_id"], "group": prompt_row["group"],
           "prompt": prompt_row["prompt"], "seed": int(seed), "steps": GEN["steps"],
           "guidance_scale": GEN["guidance_scale"], "scheduler": GEN["scheduler"],
           "lora_scale": GEN["lora_scale"] if model != "base" else None, "nsfw_flagged": nsfw_flagged}
    assert tuple(rec) == IMAGE_METADATA_FIELDS
    return rec

def is_flagged(v):
    """True only for a real positive safety-checker flag (robust to NaN/None/str after a CSV round-trip)."""
    return v is True or (isinstance(v, (bool, np.bool_)) and bool(v)) or (isinstance(v, str) and v.strip().lower() == "true")

def build_pipeline(lora_dir=None, adapter_name=None, device=DEVICE):
    """SD 1.5 pipeline with the shared sampler; the built-in safety checker is the NSFW filter (D-056)."""
    from diffusers import DPMSolverMultistepScheduler, StableDiffusionPipeline
    pipe = StableDiffusionPipeline.from_pretrained(BASE_MODEL_ID, torch_dtype=torch_dtype(WEIGHT_DTYPE), **_model_kw())
    if not GEN["safety_checker"]:
        pipe.safety_checker = None
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    if lora_dir is not None:
        pipe.load_lora_weights(str(lora_dir), weight_name="pytorch_lora_weights.safetensors", adapter_name=adapter_name)
        if adapter_name not in pipe.get_active_adapters():
            raise RuntimeError(f"LoRA adapter {adapter_name} did not become active after loading.")
    pipe.set_progress_bar_config(disable=True)
    return pipe.to(device)

@torch.no_grad()
def generate(pipe, prompt, seed, lora=False):
    """One image with the shared settings; returns (image, nsfw_flag)."""
    out = pipe(prompt, num_inference_steps=GEN["steps"], guidance_scale=GEN["guidance_scale"],
               height=GEN["height"], width=GEN["width"], negative_prompt=GEN["negative_prompt"],
               generator=torch.Generator(device=pipe.device).manual_seed(seed),
               cross_attention_kwargs={"scale": GEN["lora_scale"]} if lora else None)
    flags = out.nsfw_content_detected
    return out.images[0], (bool(flags[0]) if flags is not None else None)

def generation_fingerprint():
    """Everything that determines the 120 images. A different adapter/prompt/setting -> a different directory."""
    return sha256_json({"gen": json.loads(json.dumps(GEN)), "base_model": BASE_MODEL_ID, "base_revision": BASE_REVISION_RESOLVED,
                        "prompts_sha256": PROMPTS_SHA256, "adapters": {r: RUN_META[r]["adapter_sha256"] for r in RUNS}})

def image_name(pid, seed):
    return f"{pid}_s{seed}.png"

def load_gen_record(gen_dir, model, pid, seed, fp):
    """A generated image counts as done only if its PNG and sidecar exist, the fingerprint matches and the hash verifies."""
    png, side = gen_dir / model / image_name(pid, seed), gen_dir / model / (image_name(pid, seed) + ".json")
    if not (png.exists() and side.exists()):
        return None
    rec = read_json(side)
    if rec.get("generation_fingerprint") != fp or rec.get("sha256") != sha256_file(png):
        raise RuntimeError(f"{png} does not match this generation configuration — never mixed; investigate.")
    return rec

def generate_all(prompts_df):
    fp = generation_fingerprint()
    gen_dir = P2_GEN_ROOT / fp[:12]
    cfg_path = gen_dir / "generation_config.json"
    cfg = {"generation_fingerprint": fp, "gen": GEN, "base_model": BASE_MODEL_ID, "base_revision": BASE_REVISION_RESOLVED,
           "prompts_sha256": PROMPTS_SHA256, "adapters": {r: RUN_META[r]["adapter_sha256"] for r in RUNS}}
    if cfg_path.exists():
        if read_json(cfg_path)["generation_fingerprint"] != fp:
            raise RuntimeError(f"{cfg_path} belongs to a different configuration.")
    else:
        write_json(cfg_path, {**cfg, "created_at": utc_now()})
    rows = list(prompts_df.itertuples(index=False))
    peak = 0
    for model in MODELS:
        todo = [(r, s) for r in rows for s in GEN["seeds"] if load_gen_record(gen_dir, model, r.prompt_id, s, fp) is None]
        if not todo:
            print(f"{model}: 40/40 images already generated — skipped.")
            continue
        release_gpu(reset_peak=True)
        lora_dir = None if model == "base" else P2_RUNS_DIR / model / "final"
        pipe = build_pipeline(lora_dir=lora_dir, adapter_name=None if model == "base" else f"run_{model}")
        (gen_dir / model).mkdir(parents=True, exist_ok=True)
        print(f"{model}: generating {len(todo)} images…")
        try:
            for r, seed in todo:
                t0 = time.perf_counter()
                img, nsfw = generate(pipe, r.prompt, seed, lora=model != "base")
                secs = time.perf_counter() - t0
                png = gen_dir / model / image_name(r.prompt_id, seed)
                tmp = png.with_name(png.name + ".tmp.png")
                img.save(tmp); os.replace(tmp, png)
                rec = {**image_metadata(model, {"prompt_id": r.prompt_id, "group": r.group, "prompt": r.prompt}, seed, nsfw),
                       "file": f"{model}/{png.name}", "sha256": sha256_file(png), "generation_fingerprint": fp,
                       "model_id": BASE_MODEL_ID if model == "base" else f"{BASE_MODEL_ID} + LoRA run {model}",
                       "base_revision": BASE_REVISION_RESOLVED,
                       "adapter_sha256": None if model == "base" else RUN_META[model]["adapter_sha256"],
                       "source_idx": r.source_idx, "height": GEN["height"], "width": GEN["width"],
                       "negative_prompt": GEN["negative_prompt"], "safety_checker": GEN["safety_checker"],
                       "seconds": round(secs, 3), "experiment_id": EXPERIMENT_ID, "created_at": utc_now()}
                write_json(png.with_name(png.name + ".json"), rec)
        except torch.cuda.OutOfMemoryError as e:
            raise RuntimeError(oom_help(f"generation ({model})")) from e
        peak = max(peak, gpu_mem("gen")["peak_allocated_mb"])
        del pipe; release_gpu()
    records = [load_gen_record(gen_dir, m, r.prompt_id, s, fp) for m in MODELS for r in rows for s in GEN["seeds"]]
    if any(x is None for x in records) or len(records) != EXPECTED_IMAGES:
        raise RuntimeError(f"Expected {EXPECTED_IMAGES} complete images, found {sum(x is not None for x in records)}.")
    df = pd.DataFrame(records)
    df.to_csv(gen_dir / "metadata.csv", index=False)
    with open(gen_dir / "metadata.jsonl", "w") as f:
        for rec in records:
            f.write(json.dumps(rec, default=str) + "\n")
    return gen_dir, fp, df, peak

def load_image(gen_dir, rec):
    return Image.open(gen_dir / rec["file"]).convert("RGB")

def comparison_grid(gen_dir, gen_df, pid, tile=320):
    """Rows: seeds 0/1. Columns: base, run A (r4), run B (r64). Same prompt, seed and sampler in every column."""
    head, pad = 56, 6
    grid = Image.new("RGB", (3 * tile + 4 * pad, head + 2 * tile + 3 * pad), "white")
    d = ImageDraw.Draw(grid)
    prompt = gen_df[(gen_df.prompt_id == pid)].iloc[0]["prompt"]
    d.text((pad, 4), f"{pid}: {prompt[:110]}", fill="black")
    for c, model in enumerate(MODELS):
        d.text((pad + c * (tile + pad), 30), {"base": "base SD 1.5", "A": "run A (r=4)", "B": "run B (r=64)"}[model], fill="black")
        for rr, seed in enumerate(GEN["seeds"]):
            rec = gen_df[(gen_df.model == model) & (gen_df.prompt_id == pid) & (gen_df.seed == seed)].iloc[0]
            grid.paste(load_image(gen_dir, rec).resize((tile, tile)), (pad + c * (tile + pad), head + pad + rr * (tile + pad)))
            if is_flagged(rec["nsfw_flagged"]):
                d.text((pad + c * (tile + pad) + 4, head + pad + rr * (tile + pad) + 4), "NSFW-FILTERED", fill="red")
    return grid

def build_hf_image_folder(gen_dir, gen_df):
    """Hugging Face 'imagefolder' layout: images/<model>/<file>.png + metadata.jsonl (file_name + all metadata) + card."""
    out = P2_HF_STAGING / "images"
    if out.exists():
        shutil.rmtree(out)
    (out / "images").mkdir(parents=True)
    with open(out / "metadata.jsonl", "w") as f:
        for rec in gen_df.to_dict("records"):
            dst = out / "images" / rec["file"]
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(gen_dir / rec["file"], dst)
            f.write(json.dumps({"file_name": f"images/{rec['file']}", **rec}, default=str) + "\n")
    (out / "README.md").write_text(f"""---
license: {ADAPTER_LICENSE}
license_name: {ADAPTER_LICENSE_NAME}
task_categories: [text-to-image]
tags: [stable-diffusion, lora, generated-images, data266]
---
# DATA 266 Lab 2 Part 2 — generated evaluation images ({EXPERIMENT_ID})

{len(gen_df)} images: base SD 1.5, LoRA run A (rank 4) and run B (rank 64) × 20 frozen prompts × seeds {list(GEN['seeds'])}.
Identical sampler for all models: {GEN['scheduler']}, {GEN['steps']} steps, guidance {GEN['guidance_scale']}, {GEN['width']}×{GEN['height']}.
`metadata.jsonl` holds prompt, prompt group, seed, model/adapter hash, safety-checker flag and image hash for every file.
NSFW-flagged outputs are kept as produced by the SD 1.5 safety checker (black images) and are marked `nsfw_flagged: true`.

**License / use:** {ADAPTER_LICENSE_NOTE}
Prompts sha256: `{PROMPTS_SHA256}` · generation fingerprint: `{gen_df.iloc[0]['generation_fingerprint']}`.
""")
    return out

GRIDS_DIR = P2_RESULTS_DIR / "figures"
if DO_GENERATE:
    require_cuda("generation")
    GEN_DIR, GEN_FP, GEN_DF, GEN_PEAK_MB = generate_all(PROMPTS_DF)
    # The adapters must visibly change the output for the same prompt/seed (catches a LoRA that silently did not load)
    _p0 = GEN_DF[(GEN_DF.prompt_id == "P01") & (GEN_DF.seed == GEN["seeds"][0])].set_index("model")
    _arr = {m: np.asarray(load_image(GEN_DIR, _p0.loc[m]), dtype=np.float32) for m in MODELS}
    ADAPTER_EFFECT = {m: round(float(np.abs(_arr[m] - _arr["base"]).mean()), 3) for m in RUNS}
    if any(v == 0.0 for v in ADAPTER_EFFECT.values()):
        raise RuntimeError(f"An adapter produced an image identical to base SD ({ADAPTER_EFFECT}): LoRA not applied.")
    write_json(GEN_DIR / "generation_summary.json", {"images": len(GEN_DF), "peak_gpu_mb_this_session": GEN_PEAK_MB,
                                                      "adapter_effect_mean_abs_diff_P01": ADAPTER_EFFECT,
                                                      "nsfw_flagged": int(GEN_DF["nsfw_flagged"].map(is_flagged).sum())})
elif DO_REPORT:
    _fp = generation_fingerprint()
    GEN_DIR, GEN_FP = P2_GEN_ROOT / _fp[:12], _fp
    if not (GEN_DIR / "metadata.csv").exists():
        raise RuntimeError("REPORT_ONLY: the 120 generated images for this configuration do not exist — run FULL_RUN.")
    GEN_DF = pd.read_csv(GEN_DIR / "metadata.csv")
    if len(GEN_DF) != EXPECTED_IMAGES or set(GEN_DF["generation_fingerprint"]) != {GEN_FP}:
        raise RuntimeError("Saved generation metadata is incomplete or mixes configurations.")
else:
    GEN_DIR = GEN_DF = GEN_FP = None
    print("Generation not run in this mode.")
if GEN_DF is not None:
    GRIDS_DIR.mkdir(parents=True, exist_ok=True)
    for _pid in GRID_PROMPT_IDS:
        _g = comparison_grid(GEN_DIR, GEN_DF, _pid)
        _g.save(GRIDS_DIR / f"grid_{_pid}.png")
    display(Image.open(GRIDS_DIR / f"grid_{GRID_PROMPT_IDS[0]}.png"))
    print(f"{len(GEN_DF)} images in {GEN_DIR}; {len(GRID_PROMPT_IDS)} grids in {GRIDS_DIR} (HF image folder: section 5)")
    print("NSFW-flagged per model:", GEN_DF.groupby("model")["nsfw_flagged"].apply(lambda s: int(s.map(is_flagged).sum())).to_dict())

## 4. Evaluation (Task 2.4)

KID (20 in-domain generations vs held-out test images; noisy), CLIP score, LPIPS seed diversity, memorization (CLIP nearest training image), NSFW counts. **Primary protocol keeps every image as generated** (safety-filtered outputs are black and stay in); numbers excluding flagged images are reported separately as *sensitivity*. The final adapter is chosen by the pre-registered `FINAL_RUN_RULE` (or an explicit, recorded `FINAL_RUN`).

In [ ]:
# ===== 4. Automated evaluation: helpers + RUN (D-057, D-094) =====
RESULTS_COLUMNS = ["model", "kid_mean", "kid_std", "kid_subset_size", "clip_score_all", "clip_score_in_domain",
                   "clip_score_out_of_domain", "clip_score_compositional", "lpips_seed_diversity",
                   "human_win_rate_style", "human_win_rate_prompt", "nsfw_flagged", "gpu_hours", "peak_gpu_mb"]
RUN_TABLE_COLUMNS = ["run", "rank", "alpha", "total_params", "trainable_params", "trainable_pct", "learning_rate",
                     "steps", "effective_batch", "micro_batch", "grad_accum", "latent_mode", "peak_gpu_mb",
                     "train_time_s", "gpu_hours"]

def kid_subset_size(n_real, n_fake, requested=None):
    """KID subset size must not exceed the smaller set (20 in-domain generations -> at most 20)."""
    n = min(n_real, n_fake)
    if n < 2:
        raise ValueError("KID needs at least 2 images per set")
    return n if requested is None else min(requested, n)

def to_uint8_batch(images):
    """PIL list -> uint8 tensor [N, 3, H, W] (torchmetrics KID with normalize=False)."""
    return torch.from_numpy(np.stack([np.asarray(im.convert("RGB"), dtype=np.uint8) for im in images])).permute(0, 3, 1, 2)

def kid(real_images, fake_images, subset_size=None, device=DEVICE, seed=SPLIT_SEED):
    from torchmetrics.image.kid import KernelInceptionDistance
    torch.manual_seed(seed)                  # KID subsets are drawn with torch.randperm: seeded for reproducibility
    m = KernelInceptionDistance(subsets=KID_SUBSETS, normalize=False,
                                subset_size=kid_subset_size(len(real_images), len(fake_images), subset_size)).to(device)
    m.update(to_uint8_batch(real_images).to(device), real=True)
    m.update(to_uint8_batch(fake_images).to(device), real=False)
    mean, std = m.compute()
    del m
    return float(mean), float(std)

def clip_score_from_embeddings(img_emb, txt_emb):
    """torchmetrics CLIPScore definition per pair: 100 * max(cosine, 0)."""
    img = torch.nn.functional.normalize(img_emb.float(), dim=-1)
    txt = torch.nn.functional.normalize(txt_emb.float(), dim=-1)
    return (100.0 * (img * txt).sum(-1).clamp(min=0)).tolist()

class ClipModel:
    """One CLIP model for CLIP score and memorization embeddings (loaded once per stage, pinned revision)."""
    def __init__(self, device=DEVICE):
        from transformers import CLIPModel, CLIPProcessor
        self.device = device
        self.model = CLIPModel.from_pretrained(CLIP_MODEL_ID, revision=CLIP_REVISION_RESOLVED).to(device).eval()
        self.proc = CLIPProcessor.from_pretrained(CLIP_MODEL_ID, revision=CLIP_REVISION_RESOLVED)
    @torch.no_grad()
    def image_embeddings(self, images, batch=32):
        out = []
        for i in range(0, len(images), batch):
            x = self.proc(images=list(images[i:i + batch]), return_tensors="pt").to(self.device)
            out.append(torch.nn.functional.normalize(self.model.get_image_features(**x).float(), dim=-1))
        return torch.cat(out)
    @torch.no_grad()
    def text_embeddings(self, texts):
        x = self.proc(text=list(texts), return_tensors="pt", padding=True, truncation=True).to(self.device)
        return torch.nn.functional.normalize(self.model.get_text_features(**x).float(), dim=-1)
    def scores(self, images, prompts):
        return clip_score_from_embeddings(self.image_embeddings(images), self.text_embeddings(prompts))

def lpips_model(device=DEVICE):
    import lpips
    return lpips.LPIPS(net=LPIPS_NET, verbose=False).to(device).eval()

@torch.no_grad()
def lpips_distance(model, img_a, img_b):
    dev = next(model.parameters()).device
    return float(model(to_model_tensor(img_a)[None].to(dev), to_model_tensor(img_b)[None].to(dev)))

def nearest_training(gen_emb, train_emb):
    """Memorization check: for each generated image, the most similar training image (cosine of CLIP embeddings)."""
    sims = gen_emb @ train_emb.T
    best = sims.max(dim=1)
    return best.values.tolist(), best.indices.tolist()

def _mean(xs):
    xs = [float(x) for x in xs]
    return round(float(np.mean(xs)), 4) if xs else None

METRICS_PATH = P2_RESULTS_DIR / f"metrics_{GEN_FP[:12]}.json" if GEN_FP else None
PER_IMAGE_PATH = P2_RESULTS_DIR / f"per_image_metrics_{GEN_FP[:12]}.csv" if GEN_FP else None

def compute_metrics():
    if METRICS_PATH.exists():
        m = read_json(METRICS_PATH)
        if m["generation_fingerprint"] == GEN_FP and m["clip_model_revision"] == CLIP_REVISION_RESOLVED:
            print("Automated metrics already computed for these exact images — loaded, not recomputed.")
            return m
        raise RuntimeError(f"{METRICS_PATH} belongs to a different configuration.")
    require_cuda("metrics")
    t_start = time.perf_counter()
    recs = GEN_DF.to_dict("records")
    img = {(r["model"], r["prompt_id"], int(r["seed"])): load_image(GEN_DIR, r) for r in recs}
    flag = {(r["model"], r["prompt_id"], int(r["seed"])): is_flagged(r["nsfw_flagged"]) for r in recs}
    in_ids = list(PROMPTS_DF[PROMPTS_DF.group == "in_domain"]["prompt_id"])
    test_imgs = [Image.fromarray(a) for a in pixels_uint8(TEST_ROWS)]
    res = {"generation_fingerprint": GEN_FP, "clip_model_id": CLIP_MODEL_ID, "clip_model_revision": CLIP_REVISION_RESOLVED,
           "n_test_reference_images": len(test_imgs), "kid": {}, "kid_sensitivity_excluding_nsfw_flagged": {},
           "clip": {}, "lpips": {}, "lpips_sensitivity_excluding_nsfw_flagged": {}, "memorization": {}, "nsfw": {}}
    # --- KID: 20 in-domain generations per model vs the held-out test images (PRIMARY protocol: all images) ---
    for model in MODELS:
        keys = [(model, p, s) for p in in_ids for s in GEN["seeds"]]
        mean_, std_ = kid(test_imgs, [img[k] for k in keys])
        res["kid"][model] = {"mean": mean_, "std": std_, "n_fake": len(keys), "subset_size": kid_subset_size(len(test_imgs), len(keys))}
        kept = [img[k] for k in keys if not flag[k]]
        if len(kept) < len(keys):        # SENSITIVITY ONLY (separately labelled; never replaces the primary number)
            res["kid_sensitivity_excluding_nsfw_flagged"][model] = (
                dict(zip(("mean", "std"), kid(test_imgs, kept))) | {"n_fake": len(kept)} if len(kept) >= 2 else {"n_fake": len(kept), "note": "too few"})
        release_gpu()
    # --- CLIP score (all 120) + memorization embeddings with ONE CLIP model ---
    clip = ClipModel()
    order = list(img)
    prompt_of = {(r["model"], r["prompt_id"], int(r["seed"])): r["prompt"] for r in recs}
    scores = []
    for i in range(0, len(order), 16):
        ks = order[i:i + 16]
        scores += clip.scores([img[k] for k in ks], [prompt_of[k] for k in ks])
    score = dict(zip(order, scores))
    group_of = dict(zip(PROMPTS_DF.prompt_id, PROMPTS_DF.group))
    for model in MODELS:
        ks = [k for k in order if k[0] == model]
        res["clip"][model] = {"all": _mean(score[k] for k in ks),
                              **{g: _mean(score[k] for k in ks if group_of[k[1]] == g) for g in N_PROMPTS},
                              "all_excluding_nsfw_flagged_SENSITIVITY": _mean(score[k] for k in ks if not flag[k])}
        res["nsfw"][model] = {"flagged": int(sum(flag[k] for k in ks)),
                              **{g: int(sum(flag[k] for k in ks if group_of[k[1]] == g)) for g in N_PROMPTS}}
    train_pixels = TRAIN_PIXELS if globals().get("TRAIN_PIXELS") is not None else pixels_uint8(TRAIN_ROWS)
    train_emb = clip.image_embeddings([Image.fromarray(a) for a in train_pixels])
    test_emb = clip.image_embeddings(test_imgs)
    t_sims, _ = nearest_training(test_emb, train_emb)
    res["memorization_reference_test_vs_train"] = {"top1_cos_mean": _mean(t_sims), "top1_cos_max": round(max(t_sims), 4),
                                                   "note": "held-out REAL test images vs their nearest training image: the natural similarity level"}
    for model in MODELS:
        keys = [(model, p, GEN["seeds"][0]) for p in in_ids][:MEMORIZATION_N]
        sims, idx = nearest_training(clip.image_embeddings([img[k] for k in keys]), train_emb)
        res["memorization"][model] = {"top1_cos_mean": _mean(sims), "top1_cos_max": round(max(sims), 4),
                                      "pairs": sorted([{"prompt_id": k[1], "seed": k[2], "train_idx": int(TRAIN_ROWS[j]["idx"]),
                                                        "train_position": int(j), "cosine": round(s, 4)}
                                                       for k, s, j in zip(keys, sims, idx)], key=lambda d: -d["cosine"])}
    del clip, train_emb, test_emb; release_gpu()
    # --- LPIPS seed diversity: seed 0 vs seed 1 for each prompt ---
    lp = lpips_model()
    s0, s1 = GEN["seeds"][0], GEN["seeds"][1]
    for model in MODELS:
        d = {p: lpips_distance(lp, img[(model, p, s0)], img[(model, p, s1)]) for p in PROMPTS_DF.prompt_id}
        res["lpips"][model] = {"mean": _mean(d.values()), "std": round(float(np.std(list(d.values()))), 4),
                               **{g: _mean(v for p, v in d.items() if group_of[p] == g) for g in N_PROMPTS}}
        clean = [v for p, v in d.items() if not (flag[(model, p, s0)] or flag[(model, p, s1)])]
        if len(clean) < len(d):
            res["lpips_sensitivity_excluding_nsfw_flagged"][model] = {"mean": _mean(clean), "pairs": len(clean)}
        for p, v in d.items():
            for s in GEN["seeds"]:
                score[(model, p, s, "lpips_pair")] = v
    del lp; release_gpu()
    per_image = pd.DataFrame([{"model": k[0], "prompt_id": k[1], "seed": k[2], "group": group_of[k[1]], "clip_score": round(score[k], 4),
                               "lpips_pair_distance": round(score[(k[0], k[1], k[2], "lpips_pair")], 4), "nsfw_flagged": flag[k]}
                              for k in order])
    per_image.to_csv(PER_IMAGE_PATH, index=False)
    res["seconds"] = round(time.perf_counter() - t_start, 1)
    res["computed_at"] = utc_now()
    res["protocol"] = ("PRIMARY metrics include every generated image exactly as returned by the pipeline (safety-filtered "
                       "outputs are black images and stay in). Entries labelled SENSITIVITY exclude NSFW-flagged images "
                       "and are reported separately; they never replace the primary numbers.")
    write_json(METRICS_PATH, res)
    return res

def select_final(metrics):
    """Pre-registered rule (FINAL_RUN_RULE) unless the team set FINAL_RUN explicitly. The selection is never changed
    silently: a different choice for the same experiment raises."""
    ka, kb = metrics["kid"]["A"], metrics["kid"]["B"]
    if FINAL_RUN in RUNS:
        run, why = FINAL_RUN, "team override (FINAL_RUN set explicitly)"
    elif abs(ka["mean"] - kb["mean"]) <= max(ka["std"], kb["std"]):
        ca, cb = metrics["clip"]["A"]["in_domain"], metrics["clip"]["B"]["in_domain"]
        run = "A" if ca >= cb else "B"
        why = f"KID difference {abs(ka['mean'] - kb['mean']):.5f} <= max std -> higher in-domain CLIP ({ca} vs {cb})"
    else:
        run = "A" if ka["mean"] < kb["mean"] else "B"
        why = f"lower in-domain KID ({ka['mean']:.5f} vs {kb['mean']:.5f})"
    meta = RUN_META[run]
    pointer = {"experiment_id": EXPERIMENT_ID, "run": run, "reason": why, "rule": FINAL_RUN_RULE,
               "adapter_dir": str(P2_RUNS_DIR / run / "final"), "adapter_file": meta["adapter_file"],
               "adapter_sha256": meta["adapter_sha256"], "rank": meta["rank"], "alpha": meta["alpha"],
               "base_model_id": BASE_MODEL_ID, "base_model_revision": BASE_REVISION_RESOLVED,
               "lora_scale": GEN["lora_scale"], "generation_fingerprint": GEN_FP, "selected_at": utc_now()}
    local = P2_RUNS_DIR / "SELECTED_FINAL_ADAPTER.json"
    if local.exists():
        old = read_json(local)
        if (old["run"], old["adapter_sha256"]) != (run, meta["adapter_sha256"]):
            raise RuntimeError(f"Final adapter for {EXPERIMENT_ID} was already selected as run {old['run']}; "
                               f"the current rule/override gives run {run}. Not changed silently — resolve explicitly.")
        pointer = old
    else:
        write_json(local, pointer)
    if P2_SELECTED_POINTER.exists() and read_json(P2_SELECTED_POINTER) != pointer:
        with open(P2_WORK_DIR / "SELECTED_FINAL_ADAPTER.history.jsonl", "a") as f:
            f.write(json.dumps({"replaced_at": utc_now(), "previous": read_json(P2_SELECTED_POINTER)}) + "\n")
    write_json(P2_SELECTED_POINTER, pointer)      # Part 3 reads this pointer (local adapter, before any upload)
    return pointer

if DO_METRICS:
    METRICS = compute_metrics()
elif DO_REPORT:
    if not (METRICS_PATH and METRICS_PATH.exists()):
        raise RuntimeError("REPORT_ONLY: automated metrics are missing for these images — run FULL_RUN (it resumes).")
    METRICS = read_json(METRICS_PATH)
else:
    METRICS = None
    print("Metrics not computed in this mode.")
if METRICS is not None:
    FINAL = select_final(METRICS)
    print(json.dumps({"kid": METRICS["kid"], "clip_in_domain": {m: METRICS["clip"][m]["in_domain"] for m in MODELS},
                      "lpips": {m: METRICS["lpips"][m]["mean"] for m in MODELS}, "nsfw": METRICS["nsfw"]}, indent=1))
    print(f"FINAL ADAPTER: run {FINAL['run']} — {FINAL['reason']}")

### 4b. Blinded human evaluation (D-058)

25 (prompt, seed) pairs, final vs base, side randomised (seed 266), ≥ 3 independent raters, criteria `domain_style` and `prompt_match`. The notebook prepares the rater packet, a rating template and a **private** answer key; it reports `HUMAN_EVALUATION_PENDING` until real ratings exist in `eval/p2_human_ratings.csv`, and validates them (missing, duplicate, invalid, too few raters) before computing win rates.

In [ ]:
# ===== 4b. Blinded human evaluation: materials + scoring of REAL ratings only (D-058, D-095) =====
RATING_COLUMNS = ["rater", "pair_id", "criterion", "choice"]
CRITERION_QUESTIONS = {"domain_style": "Which image looks more like the Pokémon artwork style of the training data?",
                       "prompt_match": "Which image matches the prompt text better?"}

def draw_human_pairs(candidates, n=None, seed=None):
    """Blinded pairs: candidates = [(prompt_id, seed), ...]; final-vs-base side randomised per pair (seeded)."""
    n = HUMAN_EVAL_PAIRS if n is None else n
    rng = np.random.default_rng(HUMAN_EVAL_SEED if seed is None else seed)
    chosen = rng.choice(len(candidates), size=n, replace=False)
    final_left = rng.integers(0, 2, size=n).astype(bool)
    return [{"pair_id": f"H{i + 1:02d}", "prompt_id": candidates[c][0], "seed": int(candidates[c][1]),
             "left": "final" if fl else "base", "right": "base" if fl else "final"}
            for i, (c, fl) in enumerate(zip(chosen, final_left))]

def human_win_rate(ratings, pairs):
    """ratings: rows {rater, pair_id, criterion, choice in {left, right}} -> final-model win rate per criterion."""
    side = {p["pair_id"]: p for p in pairs}
    out = {}
    for crit in HUMAN_EVAL_CRITERIA:
        rows = [r for r in ratings if r["criterion"] == crit]
        wins = sum(side[r["pair_id"]][r["choice"]] == "final" for r in rows)
        out[crit] = {"win_rate": wins / len(rows) if rows else None, "judgments": len(rows),
                     "raters": len({r["rater"] for r in rows})}
    return out

HUMAN_DIR = P2_HUMAN_ROOT / (GEN_FP[:12] if GEN_FP else "none")
ANSWER_KEY_PATH = HUMAN_DIR / "answer_key_PRIVATE.json"      # never shown to raters; outside the repository

def prepare_human_eval(final):
    if ANSWER_KEY_PATH.exists():
        key = read_json(ANSWER_KEY_PATH)
        if (key["final_run"], key["generation_fingerprint"]) != (final["run"], GEN_FP):
            raise RuntimeError("Existing human-evaluation materials were built for a different final run/images.")
        print("Human-evaluation materials already prepared — reused (never regenerated).")
        return key
    candidates = [(p, s) for p in PROMPTS_DF.prompt_id for s in GEN["seeds"]]        # 40, deterministic order
    pairs = draw_human_pairs(candidates)
    packet = HUMAN_DIR / "rater_packet"; packet.mkdir(parents=True, exist_ok=True)
    recs = GEN_DF.set_index(["model", "prompt_id", "seed"])
    prompt_of = dict(zip(PROMPTS_DF.prompt_id, PROMPTS_DF.prompt))
    sheet = []
    for p in pairs:
        model_of = {"final": final["run"], "base": "base"}
        L = load_image(GEN_DIR, recs.loc[(model_of[p["left"]], p["prompt_id"], p["seed"])])
        R = load_image(GEN_DIR, recs.loc[(model_of[p["right"]], p["prompt_id"], p["seed"])])
        canvas = Image.new("RGB", (2 * 512 + 30, 512 + 70), "white")
        d = ImageDraw.Draw(canvas)
        d.text((10, 8), f"{p['pair_id']} — prompt: {prompt_of[p['prompt_id']][:120]}", fill="black")
        d.text((10, 40), "LEFT", fill="black"); d.text((512 + 20, 40), "RIGHT", fill="black")
        canvas.paste(L, (10, 60)); canvas.paste(R, (512 + 20, 60))
        canvas.save(packet / f"{p['pair_id']}.jpg", quality=92)
        sheet.append({"pair_id": p["pair_id"], "prompt": prompt_of[p["prompt_id"]], "image_file": f"rater_packet/{p['pair_id']}.jpg"})
    pd.DataFrame(sheet).to_csv(HUMAN_DIR / "rater_sheet.csv", index=False)
    template = [{"rater": f"RATER_{i}", "pair_id": p["pair_id"], "criterion": c, "choice": ""}
                for i in range(1, HUMAN_EVAL_MIN_RATERS + 1) for p in pairs for c in HUMAN_EVAL_CRITERIA]
    pd.DataFrame(template, columns=RATING_COLUMNS).to_csv(HUMAN_DIR / "p2_human_ratings_TEMPLATE.csv", index=False)
    (HUMAN_DIR / "INSTRUCTIONS.txt").write_text(
        "DATA 266 Lab 2 Part 2 — blinded human evaluation\n\n"
        f"Each rater judges all {len(pairs)} pairs in rater_packet/ independently, without discussing them.\n"
        "For every pair answer both questions with LEFT or RIGHT (no ties):\n"
        + "".join(f"  - {c}: {q}\n" for c, q in CRITERION_QUESTIONS.items())
        + "\nFill p2_human_ratings_TEMPLATE.csv (choice = left or right; replace RATER_n with a rater ID; at least "
          f"{HUMAN_EVAL_MIN_RATERS} raters: both team members + one outside person), save the merged file as "
          "implementation/eval/p2_human_ratings.csv and run the notebook with MODE='REPORT_ONLY'.\n"
          "Do NOT open answer_key_PRIVATE.json before all ratings are collected.\n")
    key = {"final_run": final["run"], "generation_fingerprint": GEN_FP, "pairs": pairs, "seed": HUMAN_EVAL_SEED,
           "created_at": utc_now()}
    write_json(ANSWER_KEY_PATH, key)
    return key

def load_human_ratings(pairs):
    """Returns (rows, status, problems). Never invents, imputes or repairs a judgment."""
    if not P2_HUMAN_RATINGS_PATH.exists():
        return None, "HUMAN_EVALUATION_PENDING", [f"{P2_HUMAN_RATINGS_PATH} not found"]
    df = pd.read_csv(P2_HUMAN_RATINGS_PATH, dtype=str, keep_default_na=False)
    problems = [f"missing column {c}" for c in RATING_COLUMNS if c not in df.columns]
    if problems:
        return None, "HUMAN_EVALUATION_INVALID", problems
    df = df[RATING_COLUMNS].apply(lambda s: s.str.strip())
    df["choice"], df["criterion"] = df["choice"].str.lower(), df["criterion"].str.lower()
    ids = {p["pair_id"] for p in pairs}
    if (df == "").any(axis=1).any():
        problems.append(f"{int((df == '').any(axis=1).sum())} rows with an empty field (unrated pairs?)")
    if not set(df["choice"]) <= {"left", "right"}:
        problems.append(f"invalid choices {sorted(set(df['choice']) - {'left', 'right'})}")
    if not set(df["pair_id"]) <= ids:
        problems.append(f"unknown pair_id {sorted(set(df['pair_id']) - ids)[:5]}")
    if not set(df["criterion"]) <= set(HUMAN_EVAL_CRITERIA):
        problems.append(f"unknown criterion {sorted(set(df['criterion']) - set(HUMAN_EVAL_CRITERIA))}")
    dup = df.duplicated(["rater", "pair_id", "criterion"], keep=False)
    if dup.any():
        problems.append(f"{int(dup.sum())} duplicated (rater, pair_id, criterion) rows")
    raters = sorted(set(df["rater"]) - {""})
    if len(raters) < HUMAN_EVAL_MIN_RATERS:
        problems.append(f"{len(raters)} raters < required {HUMAN_EVAL_MIN_RATERS}")
    need = len(ids) * len(HUMAN_EVAL_CRITERIA)
    incomplete = {r: int(n) for r, n in df.groupby("rater").size().items() if n != need}
    if incomplete:
        problems.append(f"raters without exactly {need} judgments: {incomplete}")
    if problems:
        return None, "HUMAN_EVALUATION_INVALID", problems
    return df.to_dict("records"), "HUMAN_EVALUATION_COMPLETE", []

def score_human(key):
    rows, status, problems = load_human_ratings(key["pairs"])
    out = {"status": status, "problems": problems, "final_run": key["final_run"], "pairs": len(key["pairs"])}
    if rows is None:
        return out
    out["win_rate"] = human_win_rate(rows, key["pairs"])
    out["per_rater"] = {r: human_win_rate([x for x in rows if x["rater"] == r], key["pairs"])
                        for r in sorted({x["rater"] for x in rows})}
    side = {p["pair_id"]: p for p in key["pairs"]}
    for crit in HUMAN_EVAL_CRITERIA:     # majority vote per pair (odd number of raters recommended)
        votes = {}
        for x in rows:
            if x["criterion"] == crit:
                votes.setdefault(x["pair_id"], []).append(side[x["pair_id"]][x["choice"]] == "final")
        maj = [np.mean(v) > 0.5 for v in votes.values() if np.mean(v) != 0.5]
        out.setdefault("majority_vote_win_rate", {})[crit] = round(float(np.mean(maj)), 4) if maj else None
        out.setdefault("unanimous_pairs", {})[crit] = int(sum(len(set(v)) == 1 for v in votes.values()))
    out["ratings_sha256"] = sha256_file(P2_HUMAN_RATINGS_PATH)
    return out

if DO_HUMAN_PREP:
    HUMAN_KEY = prepare_human_eval(FINAL)
elif DO_REPORT:
    if not ANSWER_KEY_PATH.exists():
        raise RuntimeError("REPORT_ONLY: human-evaluation materials are missing — run FULL_RUN first.")
    HUMAN_KEY = read_json(ANSWER_KEY_PATH)
else:
    HUMAN_KEY = None
HUMAN = score_human(HUMAN_KEY) if HUMAN_KEY else {"status": "NOT_PREPARED_IN_THIS_MODE"}
print(f"Human evaluation: {HUMAN['status']}" + (f" — {HUMAN.get('problems')}" if HUMAN.get("problems") else ""))
if HUMAN_KEY:
    print("Rater materials:", HUMAN_DIR)

### 4c. Report tables and figures

Run table, baseline-vs-runs results table, loss curves, grids, memorization pairs and the lowest-CLIP *failure candidates* (the failure modes themselves are judged and written by the team).

In [ ]:
# ===== 4c. Report-ready tables and figures (machine-generated from saved results only) =====
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def md_table(df):
    """Markdown table without optional dependencies (pandas.to_markdown needs tabulate)."""
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    lines += ["| " + " | ".join(str(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join(lines)

def results_table():
    rows = []
    for m in MODELS:
        win = (HUMAN.get("win_rate") or {}) if m == FINAL["run"] else {}
        pending = HUMAN["status"] if m == FINAL["run"] and HUMAN["status"] != "HUMAN_EVALUATION_COMPLETE" else None
        rows.append({"model": m, "kid_mean": round(METRICS["kid"][m]["mean"], 5), "kid_std": round(METRICS["kid"][m]["std"], 5),
                     "kid_subset_size": METRICS["kid"][m]["subset_size"], "clip_score_all": METRICS["clip"][m]["all"],
                     "clip_score_in_domain": METRICS["clip"][m]["in_domain"],
                     "clip_score_out_of_domain": METRICS["clip"][m]["out_of_domain"],
                     "clip_score_compositional": METRICS["clip"][m]["compositional"],
                     "lpips_seed_diversity": METRICS["lpips"][m]["mean"],
                     "human_win_rate_style": (win.get("domain_style") or {}).get("win_rate", pending if m == FINAL["run"] else "n/a (reference)" if m == "base" else "not rated"),
                     "human_win_rate_prompt": (win.get("prompt_match") or {}).get("win_rate", pending if m == FINAL["run"] else "n/a (reference)" if m == "base" else "not rated"),
                     "nsfw_flagged": METRICS["nsfw"][m]["flagged"],
                     "gpu_hours": RUN_META[m]["gpu_hours"] if m in RUN_META else "n/a (no training)",
                     "peak_gpu_mb": RUN_META[m]["peak_gpu_mb"] if m in RUN_META else "n/a (no training)"})
    return pd.DataFrame(rows, columns=RESULTS_COLUMNS)

def run_table():
    rows = []
    for r, m in RUN_META.items():
        rows.append({"run": r, "rank": m["rank"], "alpha": m["alpha"], "total_params": m["params"]["total"],
                     "trainable_params": m["params"]["trainable"], "trainable_pct": m["params"]["trainable_pct"],
                     "learning_rate": m["config"]["learning_rate"], "steps": m["steps"], "effective_batch": m["config"]["effective_batch"],
                     "micro_batch": m["plan"]["micro_batch"], "grad_accum": m["plan"]["grad_accum"], "latent_mode": m["config"]["latent_mode"],
                     "peak_gpu_mb": m["peak_gpu_mb"], "train_time_s": m["train_time_s"], "gpu_hours": m["gpu_hours"]})
    return pd.DataFrame(rows, columns=RUN_TABLE_COLUMNS)

def loss_figure(path):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for r in RUNS:
        h = read_json(P2_RUNS_DIR / r / "final" / "loss_history.json")
        steps = np.array([x["step"] for x in h]); loss = np.array([x["loss"] for x in h])
        ax.plot(steps, loss, alpha=0.18, lw=0.8)
        k = 50
        ax.plot(steps[k - 1:], np.convolve(loss, np.ones(k) / k, mode="valid"), lw=2,
                label=f"run {r} (rank {RUNS[r]['rank']}), 50-step mean", color=ax.lines[-1].get_color())
    ax.set_xlabel("optimizer step"); ax.set_ylabel("training loss (epsilon MSE)"); ax.set_title(f"{EXPERIMENT_ID}: LoRA training loss")
    ax.legend(); ax.grid(alpha=0.3); fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)

def memorization_figure(path, model, n_show=MEMORIZATION_SHOW):
    """Top-n most similar (generated, nearest training image) pairs for the final model."""
    pairs = METRICS["memorization"][model]["pairs"][:n_show]
    by_idx = {r["idx"]: r for r in TRAIN_ROWS} if TRAIN_ROWS else {}
    canvas = Image.new("RGB", (2 * 256 + 30, n_show * 276 + 10), "white"); d = ImageDraw.Draw(canvas)
    for i, p in enumerate(pairs):
        gen = load_image(GEN_DIR, GEN_DF[(GEN_DF.model == model) & (GEN_DF.prompt_id == p["prompt_id"]) & (GEN_DF.seed == p["seed"])].iloc[0])
        canvas.paste(gen.resize((256, 256)), (10, 10 + i * 276))
        if p["train_idx"] in by_idx:
            canvas.paste(preprocess(decode_rgb(by_idx[p["train_idx"]]["raw"])[0]).resize((256, 256)), (276, 10 + i * 276))
        d.text((10, 268 + i * 276), f"{p['prompt_id']} s{p['seed']}  vs train #{p['train_idx']}  cos={p['cosine']}", fill="black")
    canvas.save(path)

def failure_candidates(path, model, n=6):
    """Lowest-CLIP-score images of the final model: CANDIDATES for the report's failure-mode discussion
    (the failure modes themselves are judged and written by the team)."""
    pi = pd.read_csv(PER_IMAGE_PATH)
    worst = pi[pi.model == model].nsmallest(n, "clip_score")
    canvas = Image.new("RGB", (n * 260 + 10, 300), "white"); d = ImageDraw.Draw(canvas)
    for i, r in enumerate(worst.itertuples()):
        rec = GEN_DF[(GEN_DF.model == model) & (GEN_DF.prompt_id == r.prompt_id) & (GEN_DF.seed == r.seed)].iloc[0]
        canvas.paste(load_image(GEN_DIR, rec).resize((250, 250)), (10 + i * 260, 10))
        d.text((10 + i * 260, 266), f"{r.prompt_id} s{r.seed} CLIP {r.clip_score:.1f}", fill="black")
    canvas.save(path)
    return worst

if DO_REPORT:
    FIG = P2_RESULTS_DIR / "figures"; FIG.mkdir(parents=True, exist_ok=True)
    RESULTS_DF, RUN_DF = results_table(), run_table()
    RESULTS_DF.to_csv(P2_RESULTS_DIR / "results_table.csv", index=False)
    RUN_DF.to_csv(P2_RESULTS_DIR / "run_table.csv", index=False)
    (P2_RESULTS_DIR / "results_table.md").write_text(md_table(RESULTS_DF) + "\n\n" + md_table(RUN_DF) + "\n")
    loss_figure(FIG / "loss_curves.png")
    if TRAIN_ROWS:
        memorization_figure(FIG / f"memorization_top{MEMORIZATION_SHOW}_run{FINAL['run']}.png", FINAL["run"])
    WORST = failure_candidates(FIG / f"failure_candidates_run{FINAL['run']}.png", FINAL["run"])
    display(RUN_DF); display(RESULTS_DF)
    display(Image.open(FIG / "loss_curves.png"))
    REPORT_SUMMARY = {
        "experiment_id": EXPERIMENT_ID, "generated_at": utc_now(), "mode": MODE,
        "status": {"training": "COMPLETE" if TRAINING_COMPLETE else "INCOMPLETE", "images": int(len(GEN_DF)),
                   "metrics": "COMPLETE", "human_evaluation": HUMAN["status"],
                   "hf_upload": "NOT_AUTHORIZED" if not UPLOAD_TO_HF else "SEE_UPLOAD_LOG"},
        "dataset": DATASET_STATS, "final_adapter": FINAL, "results_table": RESULTS_DF.to_dict("records"),
        "run_table": RUN_DF.to_dict("records"), "human": HUMAN,
        "kid_noise_caveat": "KID uses 20 generated vs ~83 test images (subset size 20): high variance; compare with its std.",
        "clip_bias_caveat": "CLIP ViT-L/14 is the same family as the SD 1.5 text encoder: CLIP score may favour SD outputs.",
        "sensitivity_excluding_nsfw_flagged": {"kid": METRICS["kid_sensitivity_excluding_nsfw_flagged"],
                                               "lpips": METRICS["lpips_sensitivity_excluding_nsfw_flagged"],
                                               "clip": {m: METRICS["clip"][m]["all_excluding_nsfw_flagged_SENSITIVITY"] for m in MODELS}},
        "figures": sorted(str(p) for p in FIG.glob("*.png"))}
    write_json(P2_RESULTS_DIR / "report_summary.json", REPORT_SUMMARY)
    print("Report artifacts:", P2_RESULTS_DIR)
else:
    print("Report not built in this mode.")

## 5. Hugging Face artifacts (Task 2.2.3, 2.3.3)

Staging folders for run A, run B, the selected final adapter and the 120-image dataset (imagefolder + metadata), each with a card. Nothing is uploaded unless `UPLOAD_TO_HF = True`, `HF_UPLOAD_AUTHORIZATION` is set and `$HF_TOKEN` exists; repos are created private.

In [ ]:
# ===== 5. Hugging Face artifacts: model cards + staging folders; upload only with explicit authorization (D-059, D-096) =====
MODEL_CARD_FIELDS = ("repo_id", "run", "base_model", "base_model_license", "dataset_id", "dataset_license",
                     "training", "results", "limitations", "intended_use")
LIMITATIONS = ("Trained on ~750 BLIP-captioned Pokémon images for 1,500 steps: strongest on single cartoon creatures on "
               "plain backgrounds; out-of-domain and multi-object prompts are pulled toward the training style. "
               "Automated metrics use 20 in-domain images (KID is noisy) and a CLIP model related to the SD text encoder. "
               "Captions are machine-generated (BLIP) and can be wrong. Outputs may resemble copyrighted characters.")

def model_card(meta):
    """Half-page model card (C3.2): base model, data, training settings, results, limitations, license."""
    missing = [k for k in MODEL_CARD_FIELDS if k not in meta]
    if missing:
        raise ValueError(f"model card is missing {missing}")
    train = "\n".join(f"- {k}: {v}" for k, v in meta["training"].items())
    results = "\n".join(f"- {k}: {v}" for k, v in meta["results"].items())
    return f"""---
license: {ADAPTER_LICENSE}
license_name: {ADAPTER_LICENSE_NAME}
base_model: {meta["base_model"]}
datasets:
- {meta["dataset_id"]}
tags: [stable-diffusion, lora, text-to-image, diffusers]
---
# {meta["repo_id"]} — run {meta["run"]}

**Base model:** {meta["base_model"]} ({meta["base_model_license"]}) · **Data:** {meta["dataset_id"]} (dataset license: {meta["dataset_license"]})

## Training settings
{train}

## Results
{results}

## Limitations
{meta["limitations"]}

## License and intended use
{meta["intended_use"]}

## How to load
```python
pipe = StableDiffusionPipeline.from_pretrained("{BASE_MODEL_ID}", revision="{BASE_REVISION_RESOLVED}", torch_dtype=torch.float16)
pipe.load_lora_weights("<this repo or folder>", weight_name="pytorch_lora_weights.safetensors")
```
"""

def card_meta(run, repo_id):
    m = RUN_META[run]
    res = {"kid_in_domain": f"{METRICS['kid'][run]['mean']:.5f} ± {METRICS['kid'][run]['std']:.5f} (base {METRICS['kid']['base']['mean']:.5f})",
           "clip_score_in_domain": f"{METRICS['clip'][run]['in_domain']} (base {METRICS['clip']['base']['in_domain']})",
           "clip_score_all": METRICS["clip"][run]["all"], "lpips_seed_diversity": METRICS["lpips"][run]["mean"],
           "nsfw_flagged_of_40": METRICS["nsfw"][run]["flagged"]}
    if run == FINAL["run"]:
        res["human_eval"] = (json.dumps(HUMAN["win_rate"]) if HUMAN["status"] == "HUMAN_EVALUATION_COMPLETE" else HUMAN["status"])
    return {"repo_id": repo_id or f"(not published) {EXPERIMENT_ID}-run{run}", "run": run, "base_model": BASE_MODEL_ID,
            "base_model_license": BASE_MODEL_LICENSE, "dataset_id": DATASET_ID, "dataset_license": DATASET_LICENSE,
            "training": {"rank/alpha": f"{m['rank']}/{m['alpha']}", "target_modules": list(LORA_TARGET_MODULES),
                         "steps": m["steps"], "learning_rate": m["config"]["learning_rate"], "scheduler": TRAIN["lr_scheduler"],
                         "effective_batch": m["config"]["effective_batch"], "precision": "fp16 frozen weights, fp32 LoRA, AMP",
                         "train_rows": m["train_rows"], "gpu": m["gpu"], "train_time_min": round(m["train_time_s"] / 60, 1),
                         "peak_gpu_mb": m["peak_gpu_mb"], "adapter_sha256": m["adapter_sha256"],
                         "base_model_revision": m["base_model_revision"], "dataset_revision": m["dataset_revision"]},
            "results": res, "limitations": LIMITATIONS, "intended_use": ADAPTER_LICENSE_NOTE}

def stage_adapter(run, name, repo_id):
    out = P2_HF_STAGING / name
    if out.exists():
        shutil.rmtree(out)
    out.mkdir(parents=True)
    src = P2_RUNS_DIR / run / "final"
    for f in ("pytorch_lora_weights.safetensors", "adapter_metadata.json", "loss_history.json"):
        shutil.copy2(src / f, out / f)
    if sha256_file(out / "pytorch_lora_weights.safetensors") != RUN_META[run]["adapter_sha256"]:
        raise RuntimeError(f"staged adapter for run {run} does not match the trained adapter")
    card = model_card(card_meta(run, repo_id))
    if name == "final":
        card = card.replace(f"— run {run}\n", f"— run {run} (SELECTED FINAL ADAPTER: {FINAL['reason']})\n", 1)
    (out / "README.md").write_text(card)
    return out

def upload_artifacts(staged):
    """Runs ONLY with UPLOAD_TO_HF=True, the exact authorization phrase, repo names and $HF_TOKEN. Repos are private."""
    if not UPLOAD_TO_HF:
        return {"status": "NOT_AUTHORIZED", "detail": "UPLOAD_TO_HF is False"}
    if HF_UPLOAD_AUTHORIZATION != "UPLOAD AUTHORIZED BY TEAM":
        return {"status": "NOT_AUTHORIZED", "detail": "HF_UPLOAD_AUTHORIZATION phrase not set"}
    token = os.environ.get(HF_TOKEN_ENV)
    if not token:
        return {"status": "BLOCKED", "detail": f"${HF_TOKEN_ENV} is not set in the kernel environment"}
    from huggingface_hub import HfApi
    api, log = HfApi(token=token), []
    for name, (folder, repo_id, repo_type) in staged.items():
        if not repo_id:
            log.append({"artifact": name, "status": "SKIPPED", "detail": "no repo id configured"}); continue
        api.create_repo(repo_id, private=HF_PRIVATE_REPOS, repo_type=repo_type, exist_ok=True)
        info = api.upload_folder(repo_id=repo_id, folder_path=str(folder), repo_type=repo_type,
                                 commit_message=f"{EXPERIMENT_ID}: {name}")
        log.append({"artifact": name, "repo_id": repo_id, "repo_type": repo_type, "status": "UPLOADED",
                    "commit": getattr(info, "oid", None), "private": HF_PRIVATE_REPOS, "at": utc_now()})
    write_json(P2_RESULTS_DIR / "hf_upload_log.json", log)
    return {"status": "DONE", "log": log}

if DO_REPORT:
    STAGED = {"run_A": (stage_adapter("A", "run_A", HF_REPO_RUN_A), HF_REPO_RUN_A, "model"),
              "run_B": (stage_adapter("B", "run_B", HF_REPO_RUN_B), HF_REPO_RUN_B, "model"),
              "final": (stage_adapter(FINAL["run"], "final", HF_REPO_FINAL), HF_REPO_FINAL, "model"),
              "images": (build_hf_image_folder(GEN_DIR, GEN_DF), HF_REPO_IMAGES, "dataset")}
    if HF_REPO_FINAL and HF_REPO_FINAL == {"A": HF_REPO_RUN_A, "B": HF_REPO_RUN_B}[FINAL["run"]]:
        STAGED.pop("final")                          # the final adapter IS that run's repo: no duplicate upload
    print("Staged for Hugging Face (nothing uploaded yet):", {k: str(v[0]) for k, v in STAGED.items()})
    print((P2_HF_STAGING / "final" / "README.md").read_text()[:1500])
    HF_UPLOAD = upload_artifacts(STAGED)
    print("Upload:", HF_UPLOAD["status"], HF_UPLOAD.get("detail", ""))
else:
    HF_UPLOAD = {"status": "NOT_RUN_IN_THIS_MODE"}

## 6. Safety analysis (C4)

License record (dataset CC BY-NC-SA 4.0, copyrighted characters → non-commercial research only; base model OpenRAIL-M), NSFW-filter counts, memorization check with the real test-vs-train similarity as reference, and the final run status record.

In [ ]:
# ===== 6. Safety analysis (C4) + final status record =====
LICENSE_RECORD = {
    "dataset": {"id": DATASET_ID, "revision": DATASET_REVISION_RESOLVED, "license": DATASET_LICENSE,
                "license_stated_on_card_at_revision": SPLIT_MANIFEST.get("license_stated"),
                "obligations": "BY: credit the dataset; NC: non-commercial only; SA: derivatives under the same license",
                "content_note": "images depict copyrighted Pokémon characters; captions machine-generated (BLIP)"},
    "base_model": {"id": BASE_MODEL_ID, "revision": BASE_REVISION_RESOLVED, "license": BASE_MODEL_LICENSE,
                   "note": "OpenRAIL-M use-based restrictions are passed on to derivatives"},
    "metric_models": {"clip": f"{CLIP_MODEL_ID}@{CLIP_REVISION_RESOLVED} (MIT)", "kid": "torch-fidelity Inception-v3 (FID weights)",
                      "lpips": f"lpips {LPIPS_NET} (BSD-2-Clause)"},
    "adapters_and_generated_images": {"declared_license": ADAPTER_LICENSE, "license_name": ADAPTER_LICENSE_NAME,
                                      "note": ADAPTER_LICENSE_NOTE, "status": "team decision recorded in DECISIONS.md D-096"},
}
write_json(P2_RESULTS_DIR / "license_record.json", LICENSE_RECORD)

if DO_REPORT:
    nsfw_df = pd.DataFrame([{"model": m, **METRICS["nsfw"][m]} for m in MODELS])
    mem = {m: {k: METRICS["memorization"][m][k] for k in ("top1_cos_mean", "top1_cos_max")} for m in MODELS}
    SAFETY = {"nsfw_filter": "SD 1.5 safety checker kept ON for every image (flagged images are returned black and stay "
                             "in the primary metrics; sensitivity numbers exclude them)",
              "nsfw_flagged": nsfw_df.to_dict("records"),
              "memorization": {"method": f"CLIP image cosine of {MEMORIZATION_N} in-domain seed-{GEN['seeds'][0]} generations vs "
                                         f"all {len(SPLIT_MANIFEST['train_ids'])} training images",
                               "per_model": mem, "reference_test_vs_train": METRICS["memorization_reference_test_vs_train"],
                               "final_top_pairs": METRICS["memorization"][FINAL["run"]]["pairs"][:MEMORIZATION_SHOW],
                               "interpretation_guide": ("a final-model top-1 cosine clearly ABOVE the real test-vs-train "
                                                        "reference max suggests near-copies; inspect the figure — the "
                                                        "judgement itself is written by the team")},
              "license": LICENSE_RECORD}
    write_json(P2_RESULTS_DIR / "safety_analysis.json", SAFETY)
    display(nsfw_df)
    print(json.dumps(SAFETY["memorization"], indent=1)[:2500])
else:
    SAFETY = None

FINAL_STATUS = {
    "mode": MODE, "experiment_id": EXPERIMENT_ID, "finished_at": utc_now(),
    "dataset_prepared": P2_SPLIT_MANIFEST_PATH.exists(), "prompts_frozen_sha256": PROMPTS_SHA256,
    "eval_files_committed": EVAL_COMMITTED, "training_complete": TRAINING_COMPLETE,
    "runs": {r: {k: RUN_META[r][k] for k in ("adapter_sha256", "train_time_s", "gpu_hours", "peak_gpu_mb")} for r in RUN_META},
    "images": int(len(GEN_DF)) if GEN_DF is not None else 0, "expected_images": EXPECTED_IMAGES,
    "metrics": "COMPLETE" if METRICS is not None else "NOT_RUN",
    "final_adapter": (FINAL["run"] if METRICS is not None else None),
    "human_evaluation": HUMAN["status"], "hf_upload": HF_UPLOAD["status"],
    "eval_hashes": {p.name: sha256_file(p) for p in (P2_PROMPTS_PATH, P2_SPLIT_MANIFEST_PATH, P2_EVAL_FREEZE_PATH) if p.exists()},
}
write_json(RUN_MANIFEST_PATH, {"environment": ENV_RECORD, "status": "FINISHED", "final_status": FINAL_STATUS})
print(json.dumps(FINAL_STATUS, indent=1, default=str))
if MODE == "SETUP_ONLY":
    print("\nSETUP_ONLY finished. Next: commit eval/ (README: 'Freeze the pre-experiment state'), then MODE='FULL_RUN'.")